# Pipeline health check

Evaluates **one deployed pipeline** (`industry` x `mode`) end to end: control-plane record, job
runs, Lakebase hot tier, LakeTS outcomes, CDF / Unity Catalog cold tier, streaming metrics and
cross-tier data quality. It is **read-only** (SELECT / DESCRIBE / SDK GET+list only), renders
**inline only** (nothing is written anywhere) and is self-contained: it does not import the app.

How to use: attach **serverless** compute (or any cluster), set the widgets, *Run all*. With
`sample=true` the whole report renders from a synthetic dataset (no workspace access needed).
`redact=true` (default) masks every host, id, email, ARN/account, bucket and user name in all
output. See `ops/README.md` for permissions, widgets and how to read the scorecard.

> Cells 2-11 only define helpers/configuration (collapsed in Jupyter). The report starts at the
> **Scorecard**, which is filled in once every section has run.

In [1]:
from __future__ import annotations

import importlib.metadata
import importlib.util
import os
import re
from dataclasses import dataclass

INDUSTRIES = ("manufacturing", "pharmaceutical", "energy", "automotive")
MODES = ("realtime", "microbatch", "zerobus")
IN_DATABRICKS = "DATABRICKS_RUNTIME_VERSION" in os.environ
_DBUTILS = globals().get("dbutils")  # only defined inside a Databricks notebook
_ENV_PREFIX = "HEALTH_"
_LOOKBACK_MIN_H, _LOOKBACK_MAX_H = 1, 720
_IDENT_RE = re.compile(r"^[A-Za-z0-9_-]{0,128}$")


@dataclass(frozen=True)
class Params:
    """Validated notebook parameters (one pipeline per run)."""

    industry: str
    mode: str
    setup_id: str
    lookback_hours: int
    redact: bool
    sample: bool
    app_backend_project: str
    pipelines_project: str
    uc_catalog: str
    uc_schema: str
    warehouse_id: str


def _declare_widgets() -> None:
    """Create the widgets (no-op outside Databricks)."""
    if _DBUTILS is None:
        return
    w = _DBUTILS.widgets
    w.dropdown("industry", INDUSTRIES[0], list(INDUSTRIES), "Industry")
    w.dropdown("mode", MODES[0], list(MODES), "Mode")
    w.text("setup_id", "", "Setup id (empty = latest deployed for industry+mode)")
    w.text("lookback_hours", "24", "Lookback (hours)")
    w.dropdown("redact", "true", ["true", "false"], "Redact identifiers")
    w.dropdown("sample", "false", ["true", "false"], "Sample data (no workspace access)")
    w.text("app_backend_project", "", "Advanced: app-backend Lakebase project (auto-discover)")
    w.text("pipelines_project", "", "Advanced: pipelines Lakebase project (auto-discover)")
    w.text("uc_catalog", "", "Advanced: UC catalog (derived from the ledger)")
    w.text("uc_schema", "", "Advanced: UC schema (derived from the ledger)")
    w.text("warehouse_id", "", "Advanced: SQL warehouse id (from the setup)")


def _raw_param(name: str, default: str) -> str:
    """Widget value, else ``HEALTH_<NAME>`` env var, else ``default``."""
    if _DBUTILS is not None:
        try:
            return str(_DBUTILS.widgets.get(name))
        except Exception:  # widget not declared (e.g. job parameters) -> fall through
            pass
    return os.environ.get(_ENV_PREFIX + name.upper(), default)


def _as_bool(text: str, default: bool) -> bool:
    value = text.strip().lower()
    if value in {"1", "true", "yes", "y", "on"}:
        return True
    if value in {"0", "false", "no", "n", "off"}:
        return False
    return default


def _choice(name: str, allowed: tuple[str, ...], default: str, notes: list[str]) -> str:
    value = _raw_param(name, default).strip().lower()
    if value in allowed:
        return value
    notes.append(f"{name}={value!r} is not one of {allowed}; using {default!r}")
    return default


def _ident(name: str, notes: list[str]) -> str:
    value = _raw_param(name, "").strip()
    if _IDENT_RE.match(value):
        return value
    notes.append(f"{name} contains unsupported characters and was ignored")
    return ""


def load_params() -> tuple[Params, list[str]]:
    """Read + validate every widget; invalid values fall back to safe defaults."""
    notes: list[str] = []
    try:
        lookback = int(float(_raw_param("lookback_hours", "24")))
    except ValueError:
        notes.append("lookback_hours is not a number; using 24")
        lookback = 24
    clamped = max(_LOOKBACK_MIN_H, min(_LOOKBACK_MAX_H, lookback))
    if clamped != lookback:
        notes.append(f"lookback_hours clamped to {clamped}")
    # Outside Databricks no workspace is reachable, so default to the synthetic dataset.
    sample_default = "false" if IN_DATABRICKS else "true"
    params = Params(
        industry=_choice("industry", INDUSTRIES, INDUSTRIES[0], notes),
        mode=_choice("mode", MODES, MODES[0], notes),
        setup_id=_ident("setup_id", notes),
        lookback_hours=clamped,
        redact=_as_bool(_raw_param("redact", "true"), True),
        sample=_as_bool(_raw_param("sample", sample_default), not IN_DATABRICKS),
        app_backend_project=_ident("app_backend_project", notes),
        pipelines_project=_ident("pipelines_project", notes),
        uc_catalog=_ident("uc_catalog", notes),
        uc_schema=_ident("uc_schema", notes),
        warehouse_id=_ident("warehouse_id", notes),
    )
    return params, notes


_MIN_SDK = (0, 133, 0)  # first release line this notebook was verified against (has ``postgres``)


def missing_requirements() -> list[str]:
    """pip requirements the live (non-sample) mode needs but the compute lacks."""
    missing: list[str] = []
    try:
        parts = re.findall(r"\d+", importlib.metadata.version("databricks-sdk"))[:3]
        if tuple(int(p) for p in parts) < _MIN_SDK:
            missing.append("databricks-sdk>=" + ".".join(map(str, _MIN_SDK)))
    except importlib.metadata.PackageNotFoundError:
        missing.append("databricks-sdk>=" + ".".join(map(str, _MIN_SDK)))
    if importlib.util.find_spec("psycopg2") is None:
        missing.append("psycopg2-binary")
    return missing


def ensure_dependencies(params: Params) -> None:
    """Guarded fallback: install missing libraries on Databricks, then restart Python once.

    Prefer adding them in the notebook's Environment panel / job environment instead; this
    only runs on Databricks, in live mode, and only when something is missing.
    """
    if params.sample or not IN_DATABRICKS:
        return
    missing = missing_requirements()
    if not missing:
        return
    shell = globals().get("get_ipython", lambda: None)()
    if shell is None or _DBUTILS is None:
        print("Missing libraries (add them to the Environment): " + ", ".join(missing))
        return
    shell.run_line_magic("pip", "install -q " + " ".join(f"'{m}'" for m in missing))
    print(
        "Installed " + ", ".join(missing) + " - restarting Python; re-run the notebook if it stops."
    )
    _DBUTILS.library.restartPython()


_declare_widgets()
PARAMS, PARAM_NOTES = load_params()
ensure_dependencies(PARAMS)

In [2]:
from dataclasses import dataclass


@dataclass(frozen=True)
class Thresholds:
    """Every PASS/WARN/FAIL boundary lives here; edit this cell to tune the report.

    ``*_warn`` / ``*_fail`` pairs: the check WARNs / FAILs once the measured value is
    beyond the boundary (greater for "bad when high", smaller for "bad when low").
    """

    # Freshness (seconds since the newest event_time), per mode: (warn, fail)
    freshness_s: tuple[tuple[str, tuple[int, int]], ...] = (
        ("realtime", (120, 600)),
        ("microbatch", (600, 1800)),
        ("zerobus", (180, 900)),
    )
    min_rows_in_window: int = 1  # FAIL below this
    rate_drop_warn_ratio: float = 0.5  # last-5-min rate / window average, WARN below
    rate_drop_fail_ratio: float = 0.1
    silent_sensor_warn_pct: float = 5.0  # sensors seen in window but silent for 10 min
    silent_sensor_fail_pct: float = 25.0
    null_pct_warn: float = 0.1  # NOT NULL columns must be 0, nullable ones use these
    null_pct_fail: float = 5.0
    duplicate_pct_warn: float = 0.01
    duplicate_pct_fail: float = 1.0
    out_of_order_pct_warn: float = 2.0
    out_of_order_pct_fail: float = 20.0
    out_of_range_pct_warn: float = 1.0  # value outside the row's own [min_value, max_value]
    out_of_range_pct_fail: float = 10.0
    roc_outlier_pct_warn: float = 0.5  # rate-of-change outliers (> roc_sigma standard deviations)
    roc_outlier_pct_fail: float = 5.0
    roc_sigma: float = 6.0
    gap_missing_pct_warn: float = 5.0  # missing 1-minute buckets (LakeTS gap-fill view)
    gap_missing_pct_fail: float = 25.0
    lb_size_gb_warn: float = 50.0
    forward_partitions_min: int = 1  # FAIL below this
    rollup_age_s_warn: int = 600
    rollup_age_s_fail: int = 3600
    rollup_pending_warn: int = 100
    lvc_lag_s_warn: int = 120
    lvc_lag_s_fail: int = 900
    tiering_idle_h_warn: float = 6.0  # eligible chunks but policy has not run for this long
    cdf_lag_s_warn: int = 300
    cdf_lag_s_fail: int = 1800
    wal_lag_mb_warn: float = 64.0
    wal_lag_mb_fail: float = 1024.0
    recon_diff_pct_warn: float = 1.0  # Lakebase vs Delta, same closed window
    recon_diff_pct_fail: float = 5.0
    recon_settle_min: int = 15  # exclude the newest minutes (CDF flush latency) from the diff
    delta_last_commit_s_warn: int = 900
    delta_last_commit_s_fail: int = 3600
    small_file_avg_mb_warn: float = 16.0
    small_file_min_files: int = 50
    metrics_fresh_s_warn: int = 600
    metrics_fresh_s_fail: int = 1800
    e2e_p99_ms_warn: float = 5_000.0
    e2e_p99_ms_fail: float = 30_000.0
    batch_ms_warn: float = 10_000.0
    batch_ms_fail: float = 60_000.0
    lag_behind_ratio_warn: float = 0.9  # processed_rps / input_rps, WARN below
    zerobus_errors_warn: int = 1
    zerobus_errors_fail: int = 25
    failed_runs_warn: int = 1  # failed runs among the last JOB_RUN_HISTORY
    failed_runs_fail: int = 3

    def freshness(self, mode: str) -> tuple[int, int]:
        return dict(self.freshness_s).get(mode, (300, 1800))


THRESHOLDS = Thresholds()

# ── Access limits (the notebook is strictly read-only) ────────────────────────
STATEMENT_TIMEOUT_MS = 10_000
HEAVY_STATEMENT_TIMEOUT_MS = (
    40_000  # window aggregates (planning alone can take 10-30s on 1000s of partitions)
)
LAKEBASE_CONNECT_TIMEOUT_S = 10
LAKEBASE_ROW_LIMIT = 2_000
UC_WAIT_TIMEOUT = "30s"
UC_ROW_LIMIT = 5_000
JOB_RUN_HISTORY = 10
ORDER_SAMPLE_ROWS = 50_000
QUALITY_WINDOW_HOURS = 6  # null/duplicate/range scans look at the newest N hours only (cost)
QUALITY_HOURS = min(PARAMS.lookback_hours, QUALITY_WINDOW_HOURS)
# Exact scans of the hot tier are bounded: when the lookback window holds more rows than this
# (estimated from partition statistics) the scans cover only the newest slice that fits.
EXACT_SCAN_MAX_ROWS = 3_000_000

# ── Naming conventions mirrored from the app (src/iot_twin) ───────────────────
LB_TABLE = "public.sensor_readings"
PG_DATABASE = "databricks_postgres"
PRIMARY_ENDPOINT, REPLICA_ENDPOINT = "primary", "replica"
APP_BACKEND_PROJECT_PREFIX = "iot-app-backend"
PIPELINES_PROJECT_PREFIX = "iot-pipelines"
APP_BACKEND_BRANCH = "production"
RTM_CONF_KEY = "spark.databricks.streaming.realTimeMode.enabled"
CDF_SHADOW_SCHEMA = "public"
CDF_SHADOW_TABLE = f"_shadow_{PARAMS.industry}_sensor_readings"
HEALTHY_DEPLOYMENT_STATUSES = frozenset({"deployed"})
ACTIVE_RUN_STATES = frozenset({"RUNNING", "PENDING", "QUEUED"})
BAD_RESULT_STATES = frozenset(
    {"FAILED", "TIMEDOUT", "INTERNAL_ERROR", "CANCELED", "UPSTREAM_FAILED"}
)

# Expected sensor schema (mirror of models/sensor_schema.py SENSOR_FIELD_MAP).
# (column, postgres type, delta type, NOT NULL in Postgres)
EXPECTED_SENSOR_COLUMNS: tuple[tuple[str, str, str, bool], ...] = (
    ("sensor_name", "text", "string", True),
    ("industry", "text", "string", True),
    ("value", "double precision", "double", True),
    ("unit", "text", "string", False),
    ("sensor_type", "text", "string", False),
    ("event_time", "timestamp with time zone", "timestamp", True),
    ("min_value", "double precision", "double", False),
    ("max_value", "double precision", "double", False),
    ("nominal_value", "double precision", "double", False),
    ("fault_active", "boolean", "boolean", False),
    ("metadata", "jsonb", "map<string,string>", False),
)
PG_ONLY_COLUMNS = ("id", "source")  # app-added Postgres columns

# Streaming-metrics columns (mirror of models/metrics_schema.py, managed table).
METRICS_COLUMNS = (
    "mode", "query_id", "query_name", "run_id", "batch_id", "timestamp", "num_input_rows",
    "input_rps", "processed_rps", "trigger_ms", "e2e_p50", "e2e_p99", "processing_p50",
    "processing_p99", "queuing_p50", "queuing_p99", "progress",
)  # fmt: skip
SPARK_METRIC_MODES = ("realtime", "microbatch")  # zerobus is not a Spark stream

# Advice attached to WARN/FAIL rows (the scorecard turns these into "recommended next steps").
NEXT_STEPS: dict[str, str] = {
    "control_plane_record": "Create/deploy the pipeline from the admin UI, or pass the right setup_id.",
    "deployment_status": "Open the admin pipeline page and re-run deploy; check the deployment status.",
    "job_last_run": "Open the job in Workflows, read the failed task output, fix and restart it.",
    "job_recent_failures": "Review the failed runs' task logs; repeated failures usually mean Kafka/auth/config drift.",
    "job_task_health": "A task crashed inside a running job: restart the job after fixing the failing task.",
    "job_compute": "Re-deploy the pipeline so the job cluster carries the expected mode configuration.",
    "cdf_enabled": "Run CDF reconciliation for the realtime deployment (managed CDF must be confirmed).",
    "lb_connect": "Check the Lakebase endpoint state and that your user has a Postgres role on the branch.",
    "lb_chronotable": "Re-run the LakeTS partition manager; the ChronoTable needs forward partitions.",
    "lb_rows_window": "No rows in the window: check that the streaming job is running and the producer emits.",
    "lb_freshness": "Hot tier is stale: check the consumer task, Kafka lag and the Lakebase write errors.",
    "lb_ingest_rate": "Ingest rate dropped versus the window average: look for consumer restarts or source stalls.",
    "lb_silent_sensors": "Some sensors stopped reporting: check the producer/simulator for those sensors.",
    "lb_nulls": "Unexpected NULLs: check the producer payload against the sensor schema.",
    "lb_duplicates": "Duplicate readings: look for replayed Kafka offsets / consumer restarts without dedup.",
    "lb_out_of_order": "Rows arrive out of event-time order: check producer clocks and partition skew.",
    "lb_size": "Hot tier is large: verify tiering/retention policies run and shorten retention if needed.",
    "lakets_installed": "Install LakeTS on the pipeline branch (lakets.sql) before relying on its features.",
    "lakets_partitions": "Deploy/run the LakeTS partition-manager job so forward partitions exist.",
    "lakets_tiering": "Tiering has not run recently: check the LakeTS Tiering job and CDF durability gate.",
    "lakets_rollups": "Refresh the rollups (LakeTS Rollup Refresh job) and clear pending invalidations.",
    "lakets_lvc": "Last-value cache is behind: check the LVC trigger and Lakebase write load.",
    "lakets_gap_fill": "Missing minute buckets: investigate sensor outages or ingest stalls.",
    "lakets_rate_of_change": "Rate-of-change outliers: inspect the flagged sensors for faults or unit changes.",
    "lakets_value_range": "Readings outside their declared min/max: recalibrate sensors or the producer ranges.",
    "lakets_cdf_state": "CDF is not streaming: re-run CDF reconciliation and check wal2delta status.",
    "lakets_ops_jobs": "Deploy the missing LakeTS ops jobs or fix the failing ones.",
    "uc_connect": "Grant CAN_USE on a SQL warehouse (or run on serverless compute) with SELECT on the schema.",
    "uc_tables": "Provision the missing industry tables (re-run industry provisioning).",
    "uc_rows_window": "No cold-tier rows in the window: check the sink job and CDF/Delta writes.",
    "uc_freshness": "Cold tier is stale: check the Delta writer / CDF sync / zerobus ingest.",
    "uc_last_commit": "No recent Delta commits: the writer is stalled or the table is idle.",
    "uc_small_files": "Run OPTIMIZE (or enable auto-compaction / predictive optimization) on the table.",
    "uc_cdf_state": "CDF binding is unhealthy: reconcile CDF and confirm wal2delta STREAMING.",
    "uc_cdf_lag": "CDF lag is high: check wal2delta throughput and the Delta sink.",
    "metrics_table": "Metrics table is missing: re-provision the industry catalog tables.",
    "metrics_freshness": "No recent metrics rows: the query listener/metrics writer is not flushing.",
    "metrics_e2e_latency": "End-to-end latency is high: check cluster sizing, trigger interval and sink speed.",
    "metrics_batch_duration": "Batches are slow: scale the cluster or reduce per-batch work.",
    "metrics_throughput": "Processing is slower than input: scale workers or tune the sink.",
    "zerobus_ingest": "Zerobus commits are stale or erroring: inspect system.lakeflow.zerobus_ingest errors.",
    "xc_reconciliation": "Lakebase and Delta disagree: check CDF lag, dropped chunks and sink errors.",
    "xc_schema_hot": "Hot-tier schema drifted from the sensor SSOT: re-apply the ChronoTable DDL.",
    "xc_schema_cold": "Cold-tier schema drifted from the sensor SSOT: fix the table DDL / writer schema.",
    "xc_duplicates_cold": "Duplicates in Delta: add merge/dedup on (sensor_name, event_time).",
    "xc_nulls_cold": "Unexpected NULLs in Delta: check the writer's schema mapping.",
    "xc_e2e_latency": "Serving-tier delay is high: see freshness and metrics sections for the slow hop.",
    "xc_sse_evidence": "App feed evidence is weak: check the app's LVC/alert pipeline.",
}

In [3]:
import html as _html
import re
import traceback
from collections.abc import Iterable

_HOST_TLDS = r"(?:com|net|org|io|dev|ai|cloud)"
_REDACTION_RULES: tuple[tuple[str, re.Pattern[str], str], ...] = (
    # Secrets first: these are masked even when redaction is switched off.
    ("dsn", re.compile(r"\b[a-z][a-z0-9+.-]*://[^\s/:@]+:[^\s/@]+@[^\s'\"<>]+", re.I), "<dsn>"),
    ("jwt", re.compile(r"\beyJ[A-Za-z0-9_-]{5,}\.[A-Za-z0-9_-]{5,}\.[A-Za-z0-9_-]*"), "<token>"),
    ("pat", re.compile(r"\b(?:dapi|dkea|dose|dsapi)[a-z0-9]{16,}\b", re.I), "<token>"),
    (
        "kv-secret",
        re.compile(
            r"(?i)\b(password|passwd|pwd|secret|client_secret|token|api[_-]?key|authorization)"
            r"(\s*[=:]\s*)(?:bearer\s+)?[^\s,;'\"<>)}\]]+"
        ),
        r"\1\2<redacted>",
    ),
    ("bearer", re.compile(r"(?i)\bbearer\s+[A-Za-z0-9._~+/=-]{12,}"), "Bearer <token>"),
)
_IDENTIFIER_RULES: tuple[tuple[str, re.Pattern[str], str], ...] = (
    (
        "arn",
        re.compile(r"\barn:aws[a-z-]*:[a-z0-9-]+:[a-z0-9-]*:\d{0,12}:[^\s'\"<>,;)]+", re.I),
        "<arn>",
    ),
    ("email", re.compile(r"\b[\w.+-]+@[\w-]+(?:\.[\w-]+)+\b"), "<email>"),
    (
        "bucket-uri",
        re.compile(r"\b(s3a?|gs|abfss?|wasbs?|r2)://[^\s/'\"<>@]+(?:@[^\s/'\"<>]+)?", re.I),
        r"\1://<bucket>",
    ),
    (
        "workspace-path",
        re.compile(r"(?P<p>/(?:Workspace/)?(?:Users|Repos|Shared/\.[a-z]+))/[^/\s'\"<>]+"),
        r"\g<p>/<user>",
    ),
    ("url-host", re.compile(r"\b(https?)://[^\s/'\"<>]+", re.I), r"\1://<host>"),
    ("lakebase-ep", re.compile(r"(?<![A-Za-z0-9_-])ep-[a-z]+-[a-z]+-[a-z0-9]{4,}"), "<host>"),
    (
        "host",
        re.compile(
            rf"(?<![\w@<>.-])(?:[A-Za-z0-9](?:[A-Za-z0-9-]*[A-Za-z0-9])?\.){{1,6}}{_HOST_TLDS}\b"
            r"(?::\d{2,5})?",
            re.I,
        ),
        "<host>",
    ),
    (
        "uuid",
        re.compile(r"\b[0-9a-f]{8}-[0-9a-f]{4}-[0-9a-f]{4}-[0-9a-f]{4}-[0-9a-f]{12}\b", re.I),
        "<uuid>",
    ),
    ("hex-id", re.compile(r"\b[0-9a-f]{32}\b", re.I), "<hex-id>"),
    ("cluster-id", re.compile(r"\b\d{4}-\d{6}-[a-z0-9]{8}\b"), "<cluster-id>"),
    ("long-id", re.compile(r"(?<![\d,.])\d{10,}(?![\d,]|\.\d)"), "<id>"),
)
_MIN_KNOWN_LEN = 3
_TRACE_MAX_CHARS = 1200


class Redactor:
    """Central masking for everything the notebook renders.

    * Secret-looking values (tokens, passwords, DSNs) are *always* masked.
    * With ``enabled`` also hosts, ids, emails, ARNs/accounts, buckets and users are masked,
      plus every exact value registered through :meth:`register` (job ids, catalog names...).
    """

    def __init__(self, enabled: bool = True) -> None:
        self.enabled = enabled
        self._known: dict[str, str] = {}

    def register(self, value: object, label: str) -> None:
        """Mask a runtime-discovered exact value (and, for emails, its user-name part)."""
        text = str(value or "").strip()
        if len(text) < _MIN_KNOWN_LEN:
            return
        self._known[text] = label
        if "@" in text:
            local = text.split("@", 1)[0]
            if len(local) >= 4:
                self._known.setdefault(local, "<user>")

    def register_all(self, values: Iterable[object], label: str) -> None:
        for value in values:
            self.register(value, label)

    def _mask_known(self, text: str) -> str:
        # Longest first so "a.b.c" is masked before its parts.
        for value in sorted(self._known, key=len, reverse=True):
            pattern = rf"(?<![A-Za-z0-9_]){re.escape(value)}(?![A-Za-z0-9_])"
            text = re.sub(pattern, self._known[value], text, flags=re.I)
        return text

    def __call__(self, value: object) -> str:
        text = value if isinstance(value, str) else str(value)
        for _name, pattern, repl in _REDACTION_RULES:
            text = pattern.sub(repl, text)
        if not self.enabled:
            return text
        text = self._mask_known(text)
        for _name, pattern, repl in _IDENTIFIER_RULES:
            text = pattern.sub(repl, text)
        return text

    def exception(self, exc: BaseException, *, with_trace: bool = False) -> str:
        """Redacted, bounded rendering of an exception (including chained causes)."""
        if with_trace:
            raw = "".join(traceback.format_exception(type(exc), exc, exc.__traceback__))
        else:
            raw = f"{type(exc).__name__}: {exc}"
            if exc.__cause__ is not None:
                raw += f" <- {type(exc.__cause__).__name__}: {exc.__cause__}"
        cleaned = self(raw)
        if len(cleaned) > _TRACE_MAX_CHARS:
            cleaned = cleaned[:_TRACE_MAX_CHARS] + " ...[truncated]"
        return cleaned

    def html(self, value: object) -> str:
        """Redact then HTML-escape plain text for embedding in markup."""
        return _html.escape(self(value))


REDACT = Redactor(enabled=True)

In [4]:
import re

_ALLOWED_FIRST_KEYWORDS = frozenset({"select", "with", "describe", "show", "explain"})
_FORBIDDEN_KEYWORDS = re.compile(
    r"\b(insert|update|delete|merge|drop|alter|create|truncate|grant|revoke|copy|call|"
    r"optimize|vacuum|refresh|restore|analyze|reindex|execute|msck)\b",
    re.I,
)
_COMMENT_RE = re.compile(r"--[^\n]*|/\*.*?\*/", re.S)
_STRING_RE = re.compile(r"'(?:[^']|'')*'")
# Server-side functions that would write or change state even inside a SELECT.
_FORBIDDEN_FUNCTIONS = re.compile(
    r"\b(pg_terminate_backend|pg_cancel_backend|pg_reload_conf|pg_switch_wal|pg_create_\w+|"
    r"pg_drop_\w+|set_config|nextval|setval|lo_\w+|dblink\w*|"
    r"lakets\.(?:create|drop|add|remove|enable|disable|run|execute|refresh|reset)\w*)\s*\(",
    re.I,
)


class UnsafeSqlError(ValueError):
    """Raised when a statement is not strictly read-only."""


def assert_read_only(sql: str) -> str:
    """Return ``sql`` unchanged if it is a single read-only statement, else raise.

    Defence in depth on top of a read-only transaction / warehouse permissions:
    one statement, starts with SELECT/WITH/DESCRIBE/SHOW/EXPLAIN, no write/DDL keyword
    outside string literals and comments, no state-changing server functions.
    """
    stripped = _STRING_RE.sub("''", _COMMENT_RE.sub(" ", sql)).strip().rstrip(";").strip()
    if not stripped:
        raise UnsafeSqlError("empty statement")
    if ";" in stripped:
        raise UnsafeSqlError("multiple statements are not allowed")
    first = stripped.split(None, 1)[0].lower()
    if first not in _ALLOWED_FIRST_KEYWORDS:
        raise UnsafeSqlError(f"statement must start with SELECT/WITH/DESCRIBE/SHOW, got {first!r}")
    if match := _FORBIDDEN_KEYWORDS.search(stripped):
        raise UnsafeSqlError(f"forbidden keyword in read-only statement: {match.group(0)!r}")
    if match := _FORBIDDEN_FUNCTIONS.search(stripped):
        raise UnsafeSqlError(f"forbidden function in read-only statement: {match.group(1)!r}")
    return sql

In [5]:
import json
import math
import sys
from datetime import UTC, datetime
from html import escape
from html.parser import HTMLParser
from typing import Any

STATUS_COLORS = {
    "PASS": "#1f8a4c", "WARN": "#b26a00", "FAIL": "#c0392b", "SKIPPED": "#6b7280", "INFO": "#2a6fdb",
}  # fmt: skip
SERIES_COLORS = ("#2a6fdb", "#e07b00", "#2e9e5b", "#8e44ad", "#c0392b")


def _display_html(markup: str) -> None:
    """Databricks: ``displayHTML``. Jupyter: a bundle with Markdown/plain-text fallbacks."""
    show = globals().get("displayHTML")
    if show is not None:
        show(escape_placeholders(markup))
        return
    from IPython.display import display

    display(text_bundle(markup), raw=True)


def text_bundle(markup: str) -> dict[str, str]:
    """``mime_bundle`` whose text versions are redacted again (defence in depth)."""
    bundle = mime_bundle(markup)
    for key in ("text/markdown", "text/plain"):
        bundle[key] = REDACT(bundle[key])
    return bundle


def emit_html(markup: str) -> None:
    """Render markup after central redaction (the only place HTML leaves the notebook)."""
    _display_html(REDACT(markup))


def emit_text(text: str) -> None:
    """Print redacted plain text."""
    print(REDACT(text))


class _RedactingStream:
    """Wrap stdout/stderr so stray print()/warnings are redacted too."""

    def __init__(self, inner: Any) -> None:
        self._inner = inner

    def write(self, data: str) -> int:
        return self._inner.write(REDACT(data) if isinstance(data, str) else data)

    def __getattr__(self, name: str) -> Any:
        return getattr(self._inner, name)


def install_output_guards() -> None:
    """Redact raw stdout/stderr and uncaught-exception tracebacks (best effort)."""
    for name in ("stdout", "stderr"):
        stream = getattr(sys, name)
        if not isinstance(stream, _RedactingStream):
            setattr(sys, name, _RedactingStream(stream))
    ip = globals().get("get_ipython", lambda: None)()
    if ip is None:
        try:
            from IPython import get_ipython

            ip = get_ipython()
        except ImportError:
            ip = None
    if ip is not None and hasattr(ip, "set_custom_exc"):

        def _handler(shell: Any, etype: Any, value: Any, tb: Any, tb_offset: Any = None) -> None:
            value.__traceback__ = tb
            emit_text("Uncaught error (redacted):\n" + REDACT.exception(value, with_trace=True))

        ip.set_custom_exc((Exception,), _handler)


# ── Formatting ────────────────────────────────────────────────────────────────
def fmt_num(value: Any, digits: int = 1) -> str:
    """Number with thousands separators; ``-`` for None/NaN."""
    if value is None or (isinstance(value, float) and math.isnan(value)):
        return "-"
    if isinstance(value, bool):
        return str(value)
    if isinstance(value, int) or (isinstance(value, float) and float(value).is_integer()):
        return f"{int(value):,}"
    if isinstance(value, float):
        return f"{value:,.{digits}f}"
    return str(value)


def fmt_bytes(value: Any) -> str:
    if value is None:
        return "-"
    size = float(value)
    for unit in ("B", "KB", "MB", "GB", "TB"):
        if abs(size) < 1024 or unit == "TB":
            return f"{size:,.0f} {unit}" if unit == "B" else f"{size:,.1f} {unit}"
        size /= 1024
    return "-"


def fmt_seconds(value: Any) -> str:
    """Compact duration: 45s, 3.2m, 5.1h, 2.4d."""
    if value is None:
        return "-"
    secs = float(value)
    for limit, unit, div in ((90, "s", 1), (5400, "m", 60), (129600, "h", 3600)):
        if abs(secs) < limit:
            return f"{secs / div:,.1f}{unit}" if unit != "s" else f"{secs:,.0f}s"
    return f"{secs / 86400:,.1f}d"


def fmt_ms(value: Any) -> str:
    if value is None:
        return "-"
    ms = float(value)
    return f"{ms:,.0f} ms" if ms < 10_000 else f"{ms / 1000:,.1f} s"


def fmt_pct(value: Any, digits: int = 2) -> str:
    return "-" if value is None else f"{float(value):,.{digits}f}%"


def fmt_ts(value: Any) -> str:
    if value is None:
        return "-"
    if isinstance(value, datetime):
        return value.astimezone(UTC).strftime("%Y-%m-%d %H:%M:%S UTC")
    return str(value)


def to_dt(value: Any) -> datetime | None:
    """Best-effort conversion of a datetime / ISO string / epoch to aware UTC."""
    if value is None:
        return None
    if isinstance(value, datetime):
        return value if value.tzinfo else value.replace(tzinfo=UTC)
    if isinstance(value, int | float):
        return datetime.fromtimestamp(float(value), UTC)
    text = str(value).strip().replace("Z", "+00:00").replace(" ", "T", 1)
    try:
        parsed = datetime.fromisoformat(text)
    except ValueError:
        return None
    return parsed if parsed.tzinfo else parsed.replace(tzinfo=UTC)


def num(value: Any) -> float | None:
    """Coerce Decimal/str/int to float; None when impossible."""
    try:
        return None if value is None else float(value)
    except (TypeError, ValueError):
        return None


# ── HTML building blocks ──────────────────────────────────────────────────────
def badge(status: str) -> str:
    color = STATUS_COLORS.get(status, STATUS_COLORS["SKIPPED"])
    return (
        f'<span style="background:{color};color:#fff;padding:1px 8px;border-radius:9px;'
        f'font-size:12px;font-weight:600">{escape(status)}</span>'
    )


def sample_banner() -> str:
    if not PARAMS.sample:
        return ""
    return (
        '<div style="border:2px dashed #b26a00;color:#b26a00;padding:6px 10px;margin:4px 0;'
        'font-weight:700">SAMPLE DATA &mdash; synthetic dataset, no workspace was contacted</div>'
    )


def html_table(
    rows: list[dict[str, Any]], columns: list[str] | None = None, max_rows: int = 40
) -> str:
    """Escaped HTML table from dict rows (values stringified via ``fmt_num``)."""
    if not rows:
        return '<p style="opacity:.7"><i>no rows</i></p>'
    cols = columns or list(rows[0])
    head = "".join(f'<th style="text-align:left;padding:2px 10px">{escape(c)}</th>' for c in cols)
    body = []
    for row in rows[:max_rows]:
        cells = "".join(
            f'<td style="padding:2px 10px;font-family:monospace;font-size:12px">'
            f"{row[c] if isinstance(row.get(c), _Raw) else escape(fmt_cell(row.get(c)))}</td>"
            for c in cols
        )
        body.append(f"<tr>{cells}</tr>")
    more = (
        f"<p><i>{len(rows) - max_rows} more rows not shown</i></p>" if len(rows) > max_rows else ""
    )
    return (
        '<table style="border-collapse:collapse;border-bottom:1px solid rgba(128,128,128,.4)">'
        f'<thead style="border-bottom:1px solid rgba(128,128,128,.6)"><tr>{head}</tr></thead>'
        f"<tbody>{''.join(body)}</tbody></table>{more}"
    )


class _Raw(str):
    """Marks a cell value as already-safe HTML."""


def raw(markup: str) -> _Raw:
    return _Raw(markup)


def fmt_cell(value: Any) -> str:
    if isinstance(value, datetime):
        return fmt_ts(value)
    return fmt_num(value) if not isinstance(value, str) else value


def kv_table(pairs: list[tuple[str, Any]]) -> str:
    rows = [{"field": k, "value": v} for k, v in pairs]
    return html_table(rows, ["field", "value"], max_rows=200)


def heading(text: str, level: int = 3) -> str:
    return f"<h{level} style='margin:10px 0 4px'>{escape(text)}</h{level}>"


def note(text: str, kind: str = "INFO") -> str:
    color = STATUS_COLORS.get(kind, STATUS_COLORS["INFO"])
    return (
        f'<div style="border-left:4px solid {color};padding:2px 10px;margin:4px 0">'
        f"{escape(text)}</div>"
    )


# ── Text fallbacks (identical copy of ops/html_fallback.py; kept in sync by a test) ──
_BLOCK_TAGS = frozenset({"div", "p", "ul", "ol", "li", "section", "pre", "thead", "tbody"})
_SKIP_TAGS = frozenset({"script", "style"})
_SVG_NS = 'xmlns="http://www.w3.org/2000/svg"'
_PURE_SVG = re.compile(r"^\s*<svg\b[^>]*>.*</svg>\s*$", re.DOTALL)
_SPACES = re.compile(r"[ \t\r\f\v]+")
_BLANKS = re.compile(r"\n{3,}")
# Characters that would otherwise turn text such as ``<host>`` into (hidden) inline HTML.
_MD_SPECIAL = re.compile(r"[\\`*<>]")
# Element names the report can emit; any other bare ``<name>`` is a redaction placeholder.
_HTML_TAGS = frozenset(
    [
        "a",
        "abbr",
        "b",
        "blockquote",
        "br",
        "caption",
        "code",
        "col",
        "colgroup",
        "dd",
        "del",
        "desc",
        "details",
        "div",
        "dl",
        "dt",
        "em",
        "g",
        "h1",
        "h2",
        "h3",
        "h4",
        "h5",
        "h6",
        "hr",
        "i",
        "img",
        "li",
        "line",
        "ol",
        "p",
        "path",
        "polyline",
        "pre",
        "rect",
        "s",
        "small",
        "span",
        "strong",
        "sub",
        "summary",
        "sup",
        "svg",
        "table",
        "tbody",
        "td",
        "text",
        "tfoot",
        "th",
        "thead",
        "title",
        "tr",
        "tspan",
        "u",
        "ul",
    ]
)
_BARE_TAG = re.compile(r"<([a-z][a-z0-9-]*)>")


class _MarkdownBuilder(HTMLParser):
    """Collects text from the report's HTML subset and lays it out as Markdown."""

    def __init__(self, markdown: bool) -> None:
        super().__init__(convert_charrefs=True)
        self._markdown = markdown
        self.out: list[str] = []
        self._skip = 0
        self._heading = 0
        self._table: list[list[str]] | None = None
        self._cell: list[str] | None = None
        self._svg: dict[str, list[str]] | None = None
        self._svg_part = ""

    def handle_starttag(self, tag: str, attrs: list[tuple[str, str | None]]) -> None:
        if tag in _SKIP_TAGS:
            self._skip += 1
        elif self._svg is not None:
            self._svg_part = tag if tag in ("title", "desc") else ""
        elif tag == "svg":
            self._svg = {"title": [], "desc": []}
        elif tag == "table":
            self._table = []
        elif tag == "tr" and self._table is not None:
            self._table.append([])
        elif tag in ("td", "th") and self._table is not None:
            self._cell = []
        elif re.fullmatch(r"h[1-6]", tag):
            self._heading = int(tag[1])
            self.out.append("\n\n" + "#" * self._heading + " ")
        elif tag == "br":
            self._write(" " if self._cell is not None else "\n")
        elif tag in _BLOCK_TAGS:
            self.out.append("\n")

    def handle_endtag(self, tag: str) -> None:
        if tag in _SKIP_TAGS:
            self._skip = max(0, self._skip - 1)
        elif tag == "svg" and self._svg is not None:
            self._flush_svg()
        elif self._svg is not None:
            self._svg_part = ""
        elif tag in ("td", "th") and self._cell is not None and self._table:
            self._table[-1].append(" ".join("".join(self._cell).split()))
            self._cell = None
        elif tag == "table" and self._table is not None:
            self.out.append("\n\n" + _pipe_table(self._table) + "\n\n")
            self._table = None
        elif re.fullmatch(r"h[1-6]", tag) and self._heading:
            self._heading = 0
            self.out.append("\n\n")
        elif tag in _BLOCK_TAGS:
            self.out.append("\n")

    def handle_data(self, data: str) -> None:
        if self._skip:
            return
        if self._svg is not None:
            if self._svg_part:
                self._svg[self._svg_part].append(data)
            return
        self._write(_MD_SPECIAL.sub(r"\\\g<0>", data) if self._markdown else data)

    def _write(self, text: str) -> None:
        if self._cell is not None:
            self._cell.append(text)
        elif self._table is None:
            self.out.append(text)

    def _flush_svg(self) -> None:
        svg, self._svg = self._svg or {}, None
        title = " ".join("".join(svg.get("title", [])).split()) or "chart"
        desc = " ".join("".join(svg.get("desc", [])).split())
        self._write(f"\n[chart] {title}" + (f": {desc}" if desc else "") + "\n")


def escape_placeholders(markup: str) -> str:
    """Escape redaction placeholders (``<host>``) that would otherwise render as empty tags."""
    return _BARE_TAG.sub(
        lambda m: m.group(0) if m.group(1) in _HTML_TAGS else f"&lt;{m.group(1)}&gt;", markup
    )


def _pipe_table(rows: list[list[str]]) -> str:
    rows = [r for r in rows if r]
    if not rows:
        return ""
    width = max(len(r) for r in rows)
    padded = [[c.replace("|", "\\|") for c in r] + [""] * (width - len(r)) for r in rows]
    lines = ["| " + " | ".join(padded[0]) + " |", "|" + "---|" * width]
    lines += ["| " + " | ".join(r) + " |" for r in padded[1:]]
    return "\n".join(lines)


def _convert(markup: str, markdown: bool) -> str:
    builder = _MarkdownBuilder(markdown)
    builder.feed(markup)
    builder.close()
    lines = [_SPACES.sub(" ", line).strip() for line in "".join(builder.out).split("\n")]
    text = _BLANKS.sub("\n\n", "\n".join(lines)).strip()
    return text or "(empty output)"


def html_to_markdown(markup: str) -> str:
    """Markdown rendering of the report's HTML (text is escaped so nothing is hidden)."""
    return _convert(markup, markdown=True)


def html_to_text(markup: str) -> str:
    """Plain-text rendering: same layout as the Markdown, without escapes."""
    return _convert(markup, markdown=False)


def standalone_svg(markup: str) -> str | None:
    """The markup as a standalone SVG document when it is exactly one ``<svg>`` element."""
    if not _PURE_SVG.match(markup) or markup.count("<svg") != 1:
        return None
    svg = markup.strip()
    return svg if _SVG_NS in svg else svg.replace("<svg", f"<svg {_SVG_NS}", 1)


def mime_bundle(markup: str) -> dict[str, str]:
    """``text/html`` plus Markdown/plain-text fallbacks (and ``image/svg+xml`` for charts)."""
    markup = escape_placeholders(markup)
    bundle = {
        "text/html": markup,
        "text/markdown": html_to_markdown(markup),
        "text/plain": html_to_text(markup),
    }
    svg = standalone_svg(markup)
    if svg is not None:
        bundle["image/svg+xml"] = svg
    return bundle


# ── Dependency-free SVG charts (render identically in every notebook UI) ──────
def _scale(lo: float, hi: float, size: float, value: float) -> float:
    return size * (value - lo) / (hi - lo) if hi > lo else size / 2


def _line_summary(
    series: dict[str, list[tuple[float, float]]],
    x_ends: tuple[str, str],
    hline: tuple[float, str] | None,
) -> str:
    """One-line text version of a line chart (shown where SVG/HTML is not rendered)."""
    parts = []
    for label, line in series.items():
        if not line:
            parts.append(f"{label}: no data")
            continue
        ys = [y for _, y in sorted(line)]
        parts.append(
            f"{label}: {len(ys)} points, min {fmt_num(min(ys))}, max {fmt_num(max(ys))}, "
            f"last {fmt_num(ys[-1])}"
        )
    span = f" ({x_ends[0]} to {x_ends[1]})" if any(x_ends) else ""
    limit = f"; {hline[1]} at {fmt_num(hline[0])}" if hline else ""
    return "; ".join(parts) + span + limit


def _bar_summary(bars: list[tuple[str, float]], y_label: str) -> str:
    """One-line text version of a bar chart."""
    values = [v for _, v in bars]
    unit = f" {y_label}" if y_label else ""
    return (
        f"{len(bars)} bars ({bars[0][0]} to {bars[-1][0]}), min {fmt_num(min(values))}, "
        f"max {fmt_num(max(values))}, last {fmt_num(values[-1])}{unit}"
    )


def svg_line_chart(
    series: dict[str, list[tuple[float, float]]],
    *,
    title: str,
    y_label: str = "",
    x_ends: tuple[str, str] = ("", ""),
    hline: tuple[float, str] | None = None,
    width: int = 720,
    height: int = 220,
) -> str:
    """Multi-series line chart. ``series`` maps label -> [(x, y)] with numeric x."""
    pts = [p for line in series.values() for p in line]
    if not pts:
        return note(f"{title}: no data in the selected window", "SKIPPED")
    left, right, top, bottom = 56, 12, 26, 28
    pw, ph = width - left - right, height - top - bottom
    x_lo, x_hi = min(p[0] for p in pts), max(p[0] for p in pts)
    y_vals = [p[1] for p in pts] + ([hline[0]] if hline else [])
    y_lo, y_hi = min(0.0, min(y_vals)), max(y_vals)
    y_hi = y_hi * 1.08 if y_hi > y_lo else y_lo + 1
    parts = [
        f'<svg xmlns="http://www.w3.org/2000/svg" role="img" aria-label="{escape(title)}" width="{width}" height="{height}" '
        f'viewBox="0 0 {width} {height}" style="max-width:100%">',
        f"<title>{escape(title)}</title><desc>{escape(_line_summary(series, x_ends, hline))}</desc>",
        f'<text x="{left}" y="16" font-size="13" font-weight="600" fill="currentColor">'
        f"{escape(title)}</text>",
    ]
    for i in range(5):
        yv = y_lo + (y_hi - y_lo) * i / 4
        y = top + ph - _scale(y_lo, y_hi, ph, yv)
        parts.append(
            f'<line x1="{left}" x2="{width - right}" y1="{y:.1f}" y2="{y:.1f}" '
            'stroke="currentColor" stroke-opacity=".15"/>'
            f'<text x="{left - 6}" y="{y + 4:.1f}" font-size="10" text-anchor="end" '
            f'fill="currentColor" fill-opacity=".7">{escape(fmt_num(yv, 1 if y_hi < 100 else 0))}</text>'
        )
    if hline:
        y = top + ph - _scale(y_lo, y_hi, ph, hline[0])
        parts.append(
            f'<line x1="{left}" x2="{width - right}" y1="{y:.1f}" y2="{y:.1f}" stroke="#c0392b" '
            'stroke-dasharray="5 4"/>'
            f'<text x="{width - right}" y="{y - 3:.1f}" font-size="10" text-anchor="end" '
            f'fill="#c0392b">{escape(hline[1])}</text>'
        )
    for idx, (label, line) in enumerate(series.items()):
        color = SERIES_COLORS[idx % len(SERIES_COLORS)]
        coords = " ".join(
            f"{left + _scale(x_lo, x_hi, pw, x):.1f},{top + ph - _scale(y_lo, y_hi, ph, y):.1f}"
            for x, y in sorted(line)
        )
        parts.append(
            f'<polyline fill="none" stroke="{color}" stroke-width="2" points="{coords}"/>'
            f'<text x="{left + 8 + idx * 150}" y="{height - 4}" font-size="11" fill="{color}">'
            f"&#9632; {escape(label)}</text>"
        )
    parts.append(
        f'<text x="{left}" y="{height - 16}" font-size="10" fill="currentColor" fill-opacity=".7">'
        f"{escape(x_ends[0])}</text>"
        f'<text x="{width - right}" y="{height - 16}" font-size="10" text-anchor="end" '
        f'fill="currentColor" fill-opacity=".7">{escape(x_ends[1])}</text>'
        f'<text x="4" y="{top + 4}" font-size="10" fill="currentColor" fill-opacity=".7">'
        f"{escape(y_label)}</text></svg>"
    )
    return "".join(parts)


def svg_bar_chart(
    bars: list[tuple[str, float]],
    *,
    title: str,
    y_label: str = "",
    width: int = 720,
    height: int = 220,
) -> str:
    """Vertical bar chart; bar labels are shown only for the first/last bar to stay legible."""
    if not bars:
        return note(f"{title}: no data in the selected window", "SKIPPED")
    left, right, top, bottom = 56, 12, 26, 28
    pw, ph = width - left - right, height - top - bottom
    y_hi = max(v for _, v in bars) * 1.08 or 1.0
    slot = pw / len(bars)
    parts = [
        f'<svg xmlns="http://www.w3.org/2000/svg" role="img" aria-label="{escape(title)}" width="{width}" height="{height}" '
        f'viewBox="0 0 {width} {height}" style="max-width:100%">',
        f"<title>{escape(title)}</title><desc>{escape(_bar_summary(bars, y_label))}</desc>",
        f'<text x="{left}" y="16" font-size="13" font-weight="600" fill="currentColor">'
        f"{escape(title)}</text>",
    ]
    for i in range(5):
        yv = y_hi * i / 4
        y = top + ph - ph * i / 4
        parts.append(
            f'<line x1="{left}" x2="{width - right}" y1="{y:.1f}" y2="{y:.1f}" '
            'stroke="currentColor" stroke-opacity=".15"/>'
            f'<text x="{left - 6}" y="{y + 4:.1f}" font-size="10" text-anchor="end" '
            f'fill="currentColor" fill-opacity=".7">{escape(fmt_num(yv, 0))}</text>'
        )
    for idx, (_label, value) in enumerate(bars):
        h = ph * value / y_hi
        parts.append(
            f'<rect x="{left + idx * slot + slot * 0.1:.1f}" y="{top + ph - h:.1f}" '
            f'width="{slot * 0.8:.1f}" height="{h:.1f}" fill="{SERIES_COLORS[0]}"/>'
        )
    parts.append(
        f'<text x="{left}" y="{height - 6}" font-size="10" fill="currentColor" fill-opacity=".7">'
        f"{escape(bars[0][0])}</text>"
        f'<text x="{width - right}" y="{height - 6}" font-size="10" text-anchor="end" '
        f'fill="currentColor" fill-opacity=".7">{escape(bars[-1][0])}</text>'
        f'<text x="4" y="{top + 4}" font-size="10" fill="currentColor" fill-opacity=".7">'
        f"{escape(y_label)}</text></svg>"
    )
    return "".join(parts)


class LiveBlock:
    """An output area that can be refreshed once the whole report has run.

    Jupyter/VS Code update the earlier output in place (``display_id``); Databricks cannot
    update a previous cell, so ``update`` renders a fresh copy at the current position.
    """

    def __init__(self) -> None:
        self._handle: Any = None

    def show(self, markup: str) -> None:
        if globals().get("displayHTML") is not None:
            emit_html(markup)
            return
        from IPython.display import display

        self._handle = display(text_bundle(REDACT(markup)), raw=True, display_id=True)

    def update(self, markup: str) -> None:
        if self._handle is None:
            emit_html(markup)
            return
        self._handle.update(text_bundle(REDACT(markup)), raw=True)

In [6]:
import re
from collections.abc import Callable
from dataclasses import dataclass, field

PASS, WARN, FAIL, SKIPPED = "PASS", "WARN", "FAIL", "SKIPPED"
_STATUS_RANK = {PASS: 0, SKIPPED: 0, WARN: 1, FAIL: 2}


@dataclass(frozen=True)
class Outcome:
    """What a check function returns; ``Report.run`` turns it into a ``CheckResult``."""

    status: str
    value: str
    threshold: str
    explanation: str
    next_step: str = ""


@dataclass(frozen=True)
class CheckResult:
    section: str
    name: str
    status: str
    value: str
    threshold: str
    explanation: str
    next_step: str = ""


class CheckSkipped(Exception):
    """A check cannot run (missing object, no access, not applicable)."""


class NotApplicable(CheckSkipped):
    """The check does not apply to this mode/configuration."""


_ERROR_CLASSES: tuple[tuple[str, re.Pattern[str]], ...] = (
    (
        "NO ACCESS",
        re.compile(
            r"permission denied|insufficient[_ ]privilege|not authorized|PERMISSION_DENIED|"
            r"does not have|forbidden|403|USE CATALOG|USE SCHEMA|SELECT on",
            re.I,
        ),
    ),
    (
        "NOT FOUND",
        re.compile(
            r"does not exist|not found|NOT_FOUND|TABLE_OR_VIEW_NOT_FOUND|SCHEMA_NOT_FOUND|"
            r"undefined_table|undefinedtable|no such|RESOURCE_DOES_NOT_EXIST|404",
            re.I,
        ),
    ),
    ("TIMEOUT", re.compile(r"timed? ?out|statement timeout|canceling statement|deadline", re.I)),
)


def classify_error(exc: BaseException) -> str:
    """NO ACCESS / NOT FOUND / TIMEOUT / ERROR for an exception (checked on its text)."""
    text = f"{type(exc).__name__} {exc}"
    for label, pattern in _ERROR_CLASSES:
        if pattern.search(text):
            return label
    return "ERROR"


def status_high(value: float | None, warn: float, fail: float) -> str:
    """Worse when higher: WARN at ``>= warn``, FAIL at ``>= fail``."""
    if value is None:
        return SKIPPED
    return FAIL if value >= fail else WARN if value >= warn else PASS


def status_low(value: float | None, warn: float, fail: float) -> str:
    """Worse when lower: WARN at ``<= warn``, FAIL at ``<= fail``."""
    if value is None:
        return SKIPPED
    return FAIL if value <= fail else WARN if value <= warn else PASS


@dataclass
class Report:
    """Collects check results and renders per-section / overall summaries."""

    results: list[CheckResult] = field(default_factory=list)
    skipped_views: list[str] = field(default_factory=list)

    def add(self, section: str, name: str, outcome: Outcome) -> CheckResult:
        fields = dict(outcome.__dict__)
        if outcome.status in (WARN, FAIL) and not outcome.next_step:
            fields["next_step"] = NEXT_STEPS.get(name, "")
        result = CheckResult(section, name, **fields)
        self.results.append(result)
        return result

    def run(self, section: str, name: str, fn: Callable[[], Outcome]) -> CheckResult:
        """Run one check; any failure becomes a SKIPPED/FAIL row instead of an exception."""
        try:
            return self.add(section, name, fn())
        except CheckSkipped as exc:
            label = "NOT APPLICABLE" if isinstance(exc, NotApplicable) else "SKIPPED"
            return self.add(section, name, Outcome(SKIPPED, "-", "-", f"{label}: {REDACT(exc)}"))
        except Exception as exc:
            label = classify_error(exc)
            detail = f"{label}: {REDACT.exception(exc)[:300]}"
            if label == "TIMEOUT":
                detail += (
                    " (reduce lookback_hours; thousands of partitions slow every hot-tier query)"
                )
            status = (
                FAIL if label in {"TIMEOUT", "ERROR"} and name.endswith("_connect") else SKIPPED
            )
            return self.add(section, name, Outcome(status, "-", "-", detail))

    def view(self, title: str, fn: Callable[[], Any]) -> None:
        """Run a purely visual block; failures render a short redacted notice."""
        try:
            fn()
        except CheckSkipped as exc:
            self.skipped_views.append(title)
            emit_html(note(f"{title}: SKIPPED - {REDACT(exc)}", "SKIPPED"))
        except Exception as exc:
            self.skipped_views.append(title)
            emit_html(
                note(f"{title}: {classify_error(exc)} - {REDACT.exception(exc)[:240]}", "SKIPPED")
            )

    def for_section(self, prefix: str) -> list[CheckResult]:
        return [r for r in self.results if r.section.startswith(prefix)]

    def overall(self) -> str:
        unresolved = [r for r in self.results if r.name == "control_plane_record"]
        if unresolved and unresolved[0].status == SKIPPED:
            return SKIPPED  # the pipeline could not be located, nothing was evaluated
        worst = max((_STATUS_RANK[r.status] for r in self.results), default=0)
        ran = any(r.status != SKIPPED for r in self.results)
        if not ran:
            return SKIPPED
        return (PASS, WARN, FAIL)[worst]

    def counts(self) -> dict[str, int]:
        return {s: sum(r.status == s for r in self.results) for s in (PASS, WARN, FAIL, SKIPPED)}


REPORT = Report()


def results_table(results: list[CheckResult]) -> str:
    rows = [
        {
            "check": r.name,
            "status": raw(badge(r.status)),
            "value": r.value,
            "threshold": r.threshold,
            "explanation": r.explanation,
        }
        for r in results
    ]
    return html_table(rows, ["check", "status", "value", "threshold", "explanation"], max_rows=100)


def finish_section(prefix: str) -> None:
    """Render the compact check table for one report section."""
    emit_html(heading("Checks in this section", 4) + results_table(REPORT.for_section(prefix)))

In [7]:
# Every statement is a SELECT / DESCRIBE; ``assert_read_only`` re-checks each one before it runs.
# Lakebase statements use psycopg2 ``%(name)s`` parameters; UC statements are ``str.format``
# templates whose identifiers are validated by ``uc_ident`` / ``quote_uc``.
_SENSOR = "public.sensor_readings"
_HOURS = "now() - make_interval(hours => %(hours)s)"
_QHOURS = "now() - make_interval(hours => %(qhours)s)"
_CHRONO = "c.schema_name = 'public' AND c.table_name = 'sensor_readings'"


def _null_counts_sql(table: str, hours_expr: str) -> str:
    cols = ", ".join(
        f"count(*) FILTER (WHERE {c} IS NULL) AS {c}__nulls" for c, *_ in EXPECTED_SENSOR_COLUMNS
    )
    return f"SELECT count(*) AS n, {cols} FROM {table} WHERE event_time >= {hours_expr}"


LAKEBASE_SQL: dict[str, tuple[str, str]] = {
    # name: (target, sql) -- target: app = control-plane Lakebase, data = pipeline replica/primary,
    # ctl = pipeline primary (LakeTS / WAL functions).
    "cp_setup_by_id": (
        "app",
        "SELECT id::text AS id, industry, flavor, source_type, status, enabled_modes, warehouse_id,"
        " cluster_spec, validation_state, created_at, updated_at FROM pipeline_setup"
        " WHERE id::text = %(setup_id)s LIMIT 1",
    ),
    "cp_setup_by_industry": (
        "app",
        "SELECT id::text AS id, industry, flavor, source_type, status, enabled_modes, warehouse_id,"
        " cluster_spec, validation_state, created_at, updated_at FROM pipeline_setup"
        " WHERE industry = %(industry)s ORDER BY updated_at DESC LIMIT 1",
    ),
    "cp_deployment": (
        "app",
        "SELECT id::text AS id, setup_id::text AS setup_id, mode, status, partition_count,"
        " cdf_enabled, lakebase_branch, lakebase_endpoints, delta_table, job_id, notebook_path,"
        " created_at, updated_at FROM pipeline_deployment"
        " WHERE setup_id::text = %(setup_id)s AND mode = %(mode)s"
        " ORDER BY updated_at DESC LIMIT 1",
    ),
    "cp_resources": (
        "app",
        "SELECT resource_type, identifier, parent_ref, teardown_order, status, created_at"
        " FROM provisioned_resource WHERE setup_id::text = %(setup_id)s"
        " ORDER BY teardown_order DESC LIMIT 200",
    ),
    "cp_alert_events": (
        "app",
        "SELECT count(*) AS events, count(*) FILTER (WHERE e.event_type = 'opened') AS opened,"
        " max(e.created_at) AS last_event_at FROM alert_event e JOIN alert a ON a.id = e.alert_id"
        f" WHERE a.industry = %(industry)s AND e.created_at >= {_HOURS}",
    ),
    "lb_latest": (
        "data",
        "SELECT event_time AS max_t, extract(epoch FROM (now() - event_time)) AS lag_s,"
        f" now() AS db_now FROM {_SENSOR} ORDER BY event_time DESC LIMIT 1",
    ),
    "lb_total_est": (
        "ctl",
        "SELECT coalesce(sum(greatest(pc.reltuples, 0)), 0)::bigint AS n_estimate,"
        " min(cm.range_start) AS min_t FROM lakets._chunk_metadata cm"
        " JOIN lakets._chronotable_registry c ON c.id = cm.chronotable_id"
        " LEFT JOIN pg_class pc ON pc.oid = to_regclass(cm.chunk_name)"
        f" WHERE {_CHRONO} AND cm.status = 'active'",
    ),
    "lb_est": (
        "ctl",
        "SELECT coalesce(sum(greatest(pc.reltuples, 0)), 0)::bigint AS n_estimate,"
        " min(cm.range_start) AS min_t FROM lakets._chunk_metadata cm"
        " JOIN lakets._chronotable_registry c ON c.id = cm.chronotable_id"
        " LEFT JOIN pg_class pc ON pc.oid = to_regclass(cm.chunk_name)"
        f" WHERE {_CHRONO} AND cm.status = 'active' AND cm.range_start <= now()"
        f" AND cm.range_end > {_HOURS}",
    ),
    "lb_est_hour": (
        "ctl",
        "SELECT date_trunc('hour', cm.range_start) AS bucket,"
        " coalesce(sum(greatest(pc.reltuples, 0)), 0)::bigint AS n"
        " FROM lakets._chunk_metadata cm JOIN lakets._chronotable_registry c"
        " ON c.id = cm.chronotable_id LEFT JOIN pg_class pc ON pc.oid = to_regclass(cm.chunk_name)"
        f" WHERE {_CHRONO} AND cm.status = 'active' AND cm.range_start <= now()"
        f" AND cm.range_end > {_HOURS} GROUP BY 1 ORDER BY 1",
    ),
    "lb_rate_minute": (
        "data",
        "SELECT date_trunc('minute', event_time) AS bucket, count(*) AS n"
        f" FROM {_SENSOR} WHERE event_time >= now() - interval '60 minutes' GROUP BY 1 ORDER BY 1",
    ),
    "lb_per_sensor": (
        "data",
        "SELECT sensor_name, max(sensor_type) AS sensor_type, max(unit) AS unit, count(*) AS n,"
        " min(event_time) AS first_event_time, max(event_time) AS last_event_time,"
        " extract(epoch FROM (now() - max(event_time))) AS silent_s"
        f" FROM {_SENSOR} WHERE event_time >= {_HOURS} GROUP BY sensor_name ORDER BY n DESC LIMIT 500",
    ),
    "lb_nulls": ("data", _null_counts_sql(_SENSOR, _QHOURS)),
    "lb_dupes": (
        "data",
        f"WITH w AS (SELECT sensor_name, event_time, value FROM {_SENSOR} WHERE event_time >= {_QHOURS})"
        " SELECT (SELECT count(*) FROM w) AS n, count(*) AS dup_groups,"
        " coalesce(sum(c - 1), 0) AS dup_rows FROM ("
        "SELECT count(*) AS c FROM w GROUP BY sensor_name, event_time, value HAVING count(*) > 1) d",
    ),
    "lb_order": (
        "data",
        "SELECT count(*) AS n, count(*) FILTER (WHERE event_time < prev_t) AS out_of_order FROM ("
        "SELECT event_time, lag(event_time) OVER (ORDER BY id) AS prev_t FROM ("
        f"SELECT id, event_time FROM {_SENSOR} WHERE event_time >= now() - interval '1 hour'"
        " ORDER BY id DESC LIMIT %(limit)s) t) x",
    ),
    "lb_range": (
        "data",
        "SELECT count(*) FILTER (WHERE min_value IS NOT NULL AND max_value IS NOT NULL) AS bounded,"
        " count(*) FILTER (WHERE min_value IS NOT NULL AND max_value IS NOT NULL"
        " AND (value < min_value OR value > max_value)) AS out_of_range"
        f" FROM {_SENSOR} WHERE event_time >= {_QHOURS}",
    ),
    "lb_roc": (
        "data",
        "WITH s AS (SELECT sensor_name, (value - lag(value) OVER w) /"
        " nullif(extract(epoch FROM event_time - lag(event_time) OVER w), 0) AS rate"
        f" FROM {_SENSOR} WHERE event_time >= now() - make_interval(hours => %(roc_hours)s)"
        " WINDOW w AS (PARTITION BY sensor_name ORDER BY event_time)),"
        " st AS (SELECT sensor_name, avg(rate) AS mu, stddev_samp(rate) AS sd FROM s GROUP BY 1)"
        " SELECT count(*) AS n, count(*) FILTER (WHERE st.sd > 0 AND"
        " abs(s.rate - st.mu) > %(sigma)s * st.sd) AS outliers, max(abs(s.rate)) AS max_abs_rate"
        " FROM s JOIN st USING (sensor_name) WHERE s.rate IS NOT NULL",
    ),
    "lb_size": (
        "ctl",
        "SELECT coalesce(sum(pc.relpages), 0)::bigint * 8192 AS heap_bytes, count(*) AS chunks"
        " FROM lakets._chunk_metadata cm JOIN lakets._chronotable_registry c"
        " ON c.id = cm.chronotable_id LEFT JOIN pg_class pc ON pc.oid = to_regclass(cm.chunk_name)"
        f" WHERE {_CHRONO} AND cm.status = 'active'",
    ),
    "lb_columns": (
        "data",
        "SELECT column_name, data_type, is_nullable FROM information_schema.columns"
        " WHERE table_schema = 'public' AND table_name = 'sensor_readings' ORDER BY ordinal_position",
    ),
    "lk_version": (
        "ctl",
        "SELECT version, installed_at FROM lakets._version ORDER BY installed_at DESC LIMIT 1",
    ),
    "lk_registry": (
        "ctl",
        "SELECT c.chunk_interval::text AS chunk_interval, c.tiering_enabled,"
        " c.retention_interval::text AS retention_interval, c.shadow_table_name, c.sync_enabled"
        f" FROM lakets._chronotable_registry c WHERE {_CHRONO}",
    ),
    "lk_chunks": (
        "ctl",
        "SELECT cm.status, count(*) AS chunks, coalesce(sum(cm.row_count), 0) AS rows,"
        " coalesce(sum(cm.size_bytes), 0) AS bytes, max(cm.tiered_at) AS last_tiered_at,"
        " min(cm.range_start) AS oldest, max(cm.range_end) AS newest,"
        " count(*) FILTER (WHERE cm.range_end > now()) AS forward,"
        f" count(*) FILTER (WHERE cm.created_at >= {_HOURS}) AS created_in_window"
        " FROM lakets._chunk_metadata cm JOIN lakets._chronotable_registry c"
        f" ON c.id = cm.chronotable_id WHERE {_CHRONO} GROUP BY cm.status",
    ),
    "lk_policies": (
        "ctl",
        "SELECT p.policy_type, p.config, p.enabled, p.last_run_at FROM lakets._policy_registry p"
        f" JOIN lakets._chronotable_registry c ON c.id = p.chronotable_id WHERE {_CHRONO}",
    ),
    "lk_tiering": (
        "ctl",
        "WITH registry AS (SELECT c.id, p.policy_type, p.enabled, p.last_run_at,"
        " CASE p.policy_type WHEN 'tiering' THEN p.config->>'after'"
        " WHEN 'tiered_retention' THEN p.config->>'tier_after' END AS tier_interval"
        " FROM lakets._chronotable_registry c LEFT JOIN LATERAL (SELECT policy_type, config,"
        " enabled, last_run_at FROM lakets._policy_registry WHERE chronotable_id = c.id"
        " AND policy_type IN ('tiering', 'tiered_retention')"
        f" ORDER BY enabled DESC NULLS LAST, id DESC LIMIT 1) p ON TRUE WHERE {_CHRONO})"
        " SELECT count(*) FILTER (WHERE cm.status = 'active') AS active,"
        " count(*) FILTER (WHERE cm.status = 'active' AND r.tier_interval IS NOT NULL"
        " AND cm.range_end <= now() - r.tier_interval::interval) AS age_eligible,"
        " count(*) FILTER (WHERE cm.status = 'tiered') AS tiered,"
        " count(*) FILTER (WHERE cm.status = 'dropped') AS dropped,"
        " r.policy_type, r.enabled, r.last_run_at,"
        " extract(epoch FROM (now() - r.last_run_at)) AS last_run_age_s"
        " FROM registry r LEFT JOIN lakets._chunk_metadata cm ON cm.chronotable_id = r.id"
        " GROUP BY r.id, r.policy_type, r.enabled, r.last_run_at, r.tier_interval",
    ),
    "lk_rollups": (
        "ctl",
        "SELECT r.name, r.bucket_interval::text AS bucket, r.watermark, r.last_refreshed_at,"
        " extract(epoch FROM (now() - r.last_refreshed_at)) AS refresh_age_s,"
        " count(i.id) AS pending_invalidations FROM lakets._rollup_registry r"
        " LEFT JOIN lakets._rollup_invalidation_log i ON i.rollup_id = r.id"
        " JOIN lakets._chronotable_registry c ON c.id = r.source_chronotable_id"
        f" WHERE {_CHRONO} GROUP BY r.id",
    ),
    "lk_lvc": (
        "ctl",
        "SELECT (SELECT count(*) FROM public._lvc_sensor_readings) AS cached_series,"
        " extract(epoch FROM ((SELECT event_time FROM public.sensor_readings"
        " ORDER BY event_time DESC LIMIT 1)"
        " - (SELECT max(event_time) FROM public._lvc_sensor_readings))) AS lag_s,"
        " extract(epoch FROM (now() - (SELECT max(last_updated) FROM public._lvc_sensor_readings)))"
        " AS update_age_s",
    ),
    "lk_gaps": (
        "data",
        "WITH obs AS (SELECT sensor_name, date_trunc('minute', event_time) AS bucket"
        f" FROM {_SENSOR} WHERE event_time >= date_trunc('minute', now()) - interval '29 minutes'"
        " GROUP BY 1, 2) SELECT count(DISTINCT sensor_name) AS sensor_count,"
        " count(*) AS observed_buckets FROM obs",
    ),
    "lk_cdf": (
        "ctl",
        "SELECT c.sync_enabled, c.shadow_table_name, lakets._cdf_committed_lsn(c.shadow_table_name)"
        " AS committed_lsn, CASE WHEN lakets._cdf_committed_lsn(c.shadow_table_name) IS NOT NULL"
        " THEN greatest(pg_wal_lsn_diff(pg_current_wal_lsn(),"
        " lakets._cdf_committed_lsn(c.shadow_table_name)), 0)::bigint END AS wal_lag_bytes"
        f" FROM lakets._chronotable_registry c WHERE {_CHRONO}",
    ),
    "lk_wal2delta": (
        "ctl",
        "SELECT (n.nspname || '.' || c.relname) AS table_name, t.status, t.committed_lsn::text"
        " AS committed_lsn FROM wal2delta.tables t JOIN pg_class c ON c.oid = t.table_oid"
        " JOIN pg_namespace n ON n.oid = c.relnamespace WHERE c.relname = %(shadow)s",
    ),
    "lb_recon": (
        "data",
        f"SELECT count(*) AS n FROM {_SENSOR} WHERE event_time >= %(t0)s::timestamptz"
        " AND event_time < %(t1)s::timestamptz",
    ),
}

_UC_WINDOW = "event_time >= current_timestamp() - INTERVAL {hours} HOURS"
_UC_QWINDOW = "event_time >= current_timestamp() - INTERVAL {qhours} HOURS"
UC_SQL: dict[str, str] = {
    "uc_tables": (
        "SELECT table_name, table_type, data_source_format, last_altered"
        " FROM {catalog}.information_schema.tables WHERE table_schema = '{schema}'"
        " AND (startswith(table_name, '{industry}_') OR table_name = '{history}')"
        " ORDER BY table_name"
    ),
    "uc_columns": (
        "SELECT column_name, data_type FROM {catalog}.information_schema.columns"
        " WHERE table_schema = '{schema}' AND table_name = '{table_name}' ORDER BY ordinal_position"
    ),
    "uc_total": "SELECT count(*) AS n, max(event_time) AS max_t FROM {table}",
    "uc_window": (
        "SELECT count(*) AS n, min(event_time) AS min_t, max(event_time) AS max_t,"
        " count(DISTINCT sensor_name) AS sensors,"
        " unix_timestamp(current_timestamp()) - unix_timestamp(max(event_time)) AS lag_s"
        f" FROM {{table}} WHERE {_UC_WINDOW}"
    ),
    "uc_nulls": (
        "SELECT count(*) AS n, "
        + ", ".join(f"count_if({c} IS NULL) AS {c}__nulls" for c, *_ in EXPECTED_SENSOR_COLUMNS)
        + f" FROM {{table}} WHERE {_UC_QWINDOW}"
    ),
    "uc_rate_hour": (
        "SELECT date_trunc('hour', event_time) AS bucket, count(*) AS n"
        f" FROM {{table}} WHERE {_UC_WINDOW} GROUP BY 1 ORDER BY 1"
    ),
    "uc_dupes": (
        f"WITH w AS (SELECT sensor_name, event_time, value FROM {{table}} WHERE {_UC_QWINDOW})"
        " SELECT (SELECT count(*) FROM w) AS n, count(*) AS dup_groups,"
        " coalesce(sum(c - 1), 0) AS dup_rows FROM ("
        "SELECT count(*) AS c FROM w GROUP BY sensor_name, event_time, value HAVING count(*) > 1) d"
    ),
    "uc_recon": (
        "SELECT count(*) AS n FROM {table} WHERE event_time >= timestamp'{t0}'"
        " AND event_time < timestamp'{t1}'"
    ),
    "uc_history": "DESCRIBE HISTORY {table} LIMIT 15",
    "uc_detail": "DESCRIBE DETAIL {table}",
    "uc_metrics": (
        "SELECT mode, batch_id, timestamp, num_input_rows, input_rps, processed_rps, trigger_ms,"
        " e2e_p50, e2e_p99, processing_p50, processing_p99, queuing_p50, queuing_p99"
        " FROM {table} WHERE mode = '{mode}'"
        " AND timestamp >= current_timestamp() - INTERVAL {hours} HOURS"
        " ORDER BY timestamp DESC LIMIT 5000"
    ),
    "uc_metrics_progress": (
        "SELECT timestamp, progress FROM {table} WHERE mode = '{mode}'"
        " ORDER BY timestamp DESC LIMIT 1"
    ),
    "uc_zerobus_summary": (
        "SELECT max(commit_time) AS last_commit_time,"
        " coalesce(sum(committed_records), 0) AS committed_records,"
        " coalesce(sum(committed_bytes), 0) AS committed_bytes,"
        " coalesce(sum(size(coalesce(errors, array()))), 0) AS error_count,"
        " unix_timestamp(now()) - unix_timestamp(max(commit_time)) AS seconds_since_commit"
        " FROM system.lakeflow.zerobus_ingest WHERE table_name = '{table_fqn}'"
        " AND commit_time >= now() - INTERVAL {hours} HOURS"
    ),
    "uc_zerobus_series": (
        "SELECT date_trunc('hour', commit_time) AS bucket,"
        " coalesce(sum(committed_records), 0) AS records,"
        " coalesce(sum(size(coalesce(errors, array()))), 0) AS errors"
        " FROM system.lakeflow.zerobus_ingest WHERE table_name = '{table_fqn}'"
        " AND commit_time >= now() - INTERVAL {hours} HOURS GROUP BY 1 ORDER BY 1"
    ),
}

In [8]:
import re
from decimal import Decimal

_UC_IDENT_RE = re.compile(r"^[A-Za-z0-9_-]+$")
_TS_RE = re.compile(r"^\d{4}-\d{2}-\d{2}[T ]\d{2}:\d{2}:\d{2}(\.\d+)?(\+00:00|Z)?$")


def uc_ident(value: str) -> str:
    """Validate a UC identifier part and backtick-quote it (hyphens are legal)."""
    if not _UC_IDENT_RE.match(value or ""):
        raise ValueError(f"unsafe Unity Catalog identifier: {value!r}")
    return f"`{value}`"


def uc_lit(value: str) -> str:
    """Validate a value that is embedded in a SQL string literal (no quotes possible)."""
    if not _UC_IDENT_RE.match(value or ""):
        raise ValueError(f"unsafe Unity Catalog name: {value!r}")
    return value


def uc_fqn(catalog: str, schema: str, table: str) -> str:
    return ".".join(uc_ident(part) for part in (catalog, schema, table))


def to_plain(value: Any) -> Any:
    """Make driver values JSON/HTML friendly (Decimal -> float, bytes -> str)."""
    if isinstance(value, Decimal):
        return float(value)
    if isinstance(value, bytes | memoryview):
        return bytes(value).decode("utf-8", "replace")
    return value


def plain_rows(rows: list[dict[str, Any]]) -> list[dict[str, Any]]:
    return [{k: to_plain(v) for k, v in row.items()} for row in rows]


class LakebaseSession:
    """One read-only psycopg2 connection (OAuth token as password, TLS, short timeouts)."""

    def __init__(self, host: str, user: str, token: str, database: str = PG_DATABASE) -> None:
        import psycopg2
        import psycopg2.extras

        self._extras = psycopg2.extras
        self._conn = psycopg2.connect(
            host=host,
            port=5432,
            dbname=database,
            user=user,
            password=token,
            sslmode="require",
            connect_timeout=LAKEBASE_CONNECT_TIMEOUT_S,
            application_name="pipeline-health-check",
            options=f"-c default_transaction_read_only=on -c statement_timeout={STATEMENT_TIMEOUT_MS}",
        )
        self._conn.set_session(readonly=True, autocommit=False)

    def query(self, sql: str, params: dict[str, Any], timeout_ms: int) -> list[dict[str, Any]]:
        assert_read_only(sql)
        try:
            with self._conn.cursor(cursor_factory=self._extras.RealDictCursor) as cur:
                cur.execute(f"SET LOCAL statement_timeout = {int(timeout_ms)}")
                cur.execute(sql, params)
                return plain_rows([dict(r) for r in cur.fetchmany(LAKEBASE_ROW_LIMIT)])
        finally:
            self._conn.rollback()  # never leave a transaction open; nothing was written

    def close(self) -> None:
        try:
            self._conn.close()
        except Exception:
            pass


def _field(obj: Any, *path: str) -> Any:
    for attr in path:
        obj = getattr(obj, attr, None)
        if obj is None:
            return None
    return obj


def _enum_text(value: Any) -> str | None:
    return None if value is None else str(getattr(value, "value", value))


def millis_to_dt(value: Any) -> datetime | None:
    number = num(value)
    return None if not number else datetime.fromtimestamp(number / 1000, UTC)


class LiveBackend:
    """Read-only access to the workspace: SDK GET/list calls, Lakebase, UC SQL."""

    label = "live workspace"

    def __init__(self, params: Params) -> None:
        from databricks.sdk import WorkspaceClient

        self.params = params
        self.ws = WorkspaceClient()
        self._sessions: dict[str, LakebaseSession] = {}
        self._targets: dict[str, tuple[str, str]] = {}  # target -> (endpoint resource, host)
        self.user = self.ws.current_user.me().user_name or ""
        self.host = str(self.ws.config.host or "")
        REDACT.register(self.user, "<user>")
        REDACT.register(self.host, "<host>")

    # ── Lakebase discovery + sessions ───────────────────────────────────────────
    def find_project(self, explicit: str, prefix: str, branch: str = "") -> str:
        """Project id: the explicit one, else the first project (name prefix) that has ``branch``."""
        if explicit:
            return explicit
        ids = sorted(p.name.split("/", 1)[1] for p in self.ws.postgres.list_projects() if p.name)
        matches = [i for i in ids if i.startswith(prefix)]
        if not matches:
            raise CheckSkipped(f"NOT FOUND: no Lakebase project starting with {prefix!r}")
        if not branch:
            return matches[0]
        for project in matches:
            parent = f"projects/{project}"
            if branch in {
                b.name.rsplit("/", 1)[-1] for b in self.ws.postgres.list_branches(parent=parent)
            }:
                return project
        raise CheckSkipped(f"NOT FOUND: no project starting with {prefix!r} has branch {branch!r}")

    def _endpoint(
        self, project: str, branch: str, endpoint_ids: tuple[str, ...]
    ) -> tuple[str, str]:
        parent = f"projects/{project}/branches/{branch}"
        names = {
            e.name.rsplit("/", 1)[-1]: e.name
            for e in self.ws.postgres.list_endpoints(parent=parent)
        }
        for endpoint_id in endpoint_ids:
            if endpoint_id in names:
                host = _field(
                    self.ws.postgres.get_endpoint(name=names[endpoint_id]),
                    "status",
                    "hosts",
                    "host",
                )
                if host:
                    return names[endpoint_id], str(host)
        raise CheckSkipped(f"NOT FOUND: no endpoint {endpoint_ids} on branch {branch!r}")

    def locate(self, target: str, project: str, branch: str, endpoint_ids: tuple[str, ...]) -> str:
        """Resolve + remember the endpoint for ``target``; returns the project id used."""
        endpoint, host = self._endpoint(project, branch, endpoint_ids)
        self._targets[target] = (endpoint, host)
        REDACT.register(host, "<host>")
        REDACT.register(project, "<lakebase-project>")
        return project

    def _session(self, target: str) -> LakebaseSession:
        if target not in self._sessions:
            if target not in self._targets:
                raise CheckSkipped(f"NOT FOUND: Lakebase target {target!r} was not resolved")
            endpoint, host = self._targets[target]
            token = self.ws.postgres.generate_database_credential(endpoint=endpoint).token
            self._sessions[target] = LakebaseSession(host, self.user, token)
        return self._sessions[target]

    def query(self, name: str, **params: Any) -> list[dict[str, Any]]:
        target, sql = LAKEBASE_SQL[name]
        heavy = LAKEBASE_SQL[name][0] == "data" and name not in {"lb_columns", "lb_rate_minute"}
        heavy = heavy or name in {"lb_size", "lb_total_est"}
        timeout = HEAVY_STATEMENT_TIMEOUT_MS if heavy else STATEMENT_TIMEOUT_MS
        try:
            return self._session(target).query(sql, params, timeout)
        except Exception as exc:
            if target != "data" or "conflict with recovery" not in str(exc):
                raise
            # Replica replay conflicted with a long read; retry once on the primary (still read-only).
            return self._session("ctl").query(sql, params, timeout)

    # ── Unity Catalog SQL (warehouse first, then spark.sql on the notebook compute) ──
    def uc(self, name: str, **fields: Any) -> list[dict[str, Any]]:
        sql = assert_read_only(UC_SQL[name].format(**fields))
        errors: list[str] = []
        if self.params.warehouse_id or self._warehouse_id:
            try:
                return self._uc_warehouse(sql)
            except Exception as exc:
                errors.append(f"warehouse: {classify_error(exc)}")
        spark = globals().get("spark")
        if spark is not None:
            return plain_rows(
                [r.asDict(recursive=True) for r in spark.sql(sql).limit(UC_ROW_LIMIT).collect()]
            )
        raise CheckSkipped(
            "NO ACCESS: no usable SQL warehouse or Spark session (" + "; ".join(errors) + ")"
        )

    _warehouse_id: str = ""

    def _uc_warehouse(self, sql: str) -> list[dict[str, Any]]:
        from databricks.sdk.service.sql import ExecuteStatementRequestOnWaitTimeout

        warehouse = self.params.warehouse_id or self._warehouse_id
        resp = self.ws.statement_execution.execute_statement(
            statement=sql,
            warehouse_id=warehouse,
            wait_timeout=UC_WAIT_TIMEOUT,
            on_wait_timeout=ExecuteStatementRequestOnWaitTimeout.CANCEL,
            row_limit=UC_ROW_LIMIT,
        )
        state = _enum_text(_field(resp, "status", "state"))
        if state != "SUCCEEDED":
            message = _field(resp, "status", "error", "message") or state
            raise RuntimeError(f"statement {state}: {message}")
        columns = [c.name for c in (_field(resp, "manifest", "schema", "columns") or [])]
        types = [
            _enum_text(c.type_name) for c in (_field(resp, "manifest", "schema", "columns") or [])
        ]
        rows = _field(resp, "result", "data_array") or []
        return [
            {c: _coerce_sql_value(v, t) for c, v, t in zip(columns, row, types, strict=True)}
            for row in rows
        ]

    def use_warehouse(self, warehouse_id: str) -> None:
        self._warehouse_id = warehouse_id

    # ── Jobs (GET / list only) ───────────────────────────────────────────────────
    def job(self, job_id: int) -> dict[str, Any]:
        settings = self.ws.jobs.get(job_id=int(job_id)).settings
        defs = list(_field(settings, "job_clusters") or [])
        defs += [
            t for t in (_field(settings, "tasks") or []) if _field(t, "new_cluster") is not None
        ]
        clusters = [
            {
                "key": getattr(c, "job_cluster_key", None) or getattr(c, "task_key", None),
                "spark_version": _field(c, "new_cluster", "spark_version"),
                "node_type": _field(c, "new_cluster", "node_type_id"),
                "workers": _field(c, "new_cluster", "num_workers"),
                "spot": _enum_text(_field(c, "new_cluster", "aws_attributes", "availability")),
                "instance_profile": _field(
                    c, "new_cluster", "aws_attributes", "instance_profile_arn"
                ),
                "runtime_engine": _enum_text(_field(c, "new_cluster", "runtime_engine")),
                "spark_conf": dict(_field(c, "new_cluster", "spark_conf") or {}),
            }
            for c in defs
        ]
        return {
            "name": _field(settings, "name"),
            "continuous": _enum_text(_field(settings, "continuous", "pause_status")),
            "is_continuous": _field(settings, "continuous") is not None,
            "serverless": bool(_field(settings, "environments")),
            "tasks": [t.task_key for t in (_field(settings, "tasks") or [])],
            "clusters": clusters,
        }

    def runs(self, job_id: int, limit: int) -> list[dict[str, Any]]:
        out = []
        for run in self.ws.jobs.list_runs(job_id=int(job_id), limit=limit, expand_tasks=False):
            state = run.state
            out.append(
                {
                    "run_id": run.run_id,
                    "life_cycle": _enum_text(_field(state, "life_cycle_state")),
                    "result": _enum_text(_field(state, "result_state")),
                    "message": _field(state, "state_message"),
                    "start": millis_to_dt(run.start_time),
                    "end": millis_to_dt(run.end_time),
                    "duration_s": (num(run.run_duration) or 0) / 1000 or None,
                }
            )
            if len(out) >= limit:
                break
        return out

    def run_tasks(self, run_id: int) -> list[dict[str, Any]]:
        run = self.ws.jobs.get_run(run_id=int(run_id))
        return [
            {
                "task": t.task_key,
                "life_cycle": _enum_text(_field(t, "state", "life_cycle_state")),
                "result": _enum_text(_field(t, "state", "result_state")),
                "message": _field(t, "state", "state_message"),
            }
            for t in (run.tasks or [])
        ]

    def jobs_named(self, name: str) -> list[dict[str, Any]]:
        return [
            {"job_id": j.job_id, "name": _field(j, "settings", "name")}
            for j in self.ws.jobs.list(name=name)
            if _field(j, "settings", "name") == name
        ]

    def warehouse_states(self) -> dict[str, str]:
        return {w.id: _enum_text(w.state) or "?" for w in self.ws.warehouses.list()}

    def close(self) -> None:
        for session in self._sessions.values():
            session.close()


def _coerce_sql_value(value: Any, type_name: str | None) -> Any:
    """Statement Execution returns strings; restore numbers/timestamps for charts and checks."""
    if value is None or type_name is None:
        return value
    if type_name in {"INT", "LONG", "SHORT", "BYTE"}:
        return int(value)
    if type_name in {"DOUBLE", "FLOAT", "DECIMAL"}:
        return float(value)
    if type_name == "TIMESTAMP":
        return to_dt(value)
    if type_name == "BOOLEAN":
        return str(value).lower() == "true"
    return value

In [9]:
import random
from datetime import timedelta

_SAMPLE_ASSETS = {
    "manufacturing": ("press_line_1", "cnc_cell_2", "conveyor_3"),
    "pharmaceutical": ("bioreactor_1", "lyophilizer_2", "fill_line_3"),
    "energy": ("turbine_1", "transformer_2", "feeder_3"),
    "automotive": ("weld_robot_1", "paint_booth_2", "assembly_3"),
}
_SAMPLE_MEASURES = ("temperature", "vibration", "pressure", "current", "speed", "humidity")
_SAMPLE_SENSORS_PER_SECOND = 1.0


class SampleBackend:
    """Deterministic synthetic dataset used when ``sample=true`` (no workspace contact).

    Identifiers below are fabricated but deliberately realistic (hosts, ids, ARNs, emails) so the
    redaction layer is exercised exactly as it is against a real workspace.
    """

    label = "SAMPLE DATA (synthetic)"
    user = "alex.morgan@acme-industrial.example.com"
    host = "https://example-ws.cloud.databricks.com"
    _SETUP_ID = "7c1e2f94-3b5a-4d68-9e0f-a1b2c3d4e5f6"
    _DEPLOYMENT_ID = "0d9f3a71-6c2e-48b5-b7a4-5e8c1f2d9a03"
    _JOB_ID = 884201937452610
    _WAREHOUSE_ID = "a1b2c3d4e5f60718"
    _HOT_HOST = "example-lakebase.database.us-east-1.cloud.databricks.com"

    def __init__(self, params: Params) -> None:
        self.params = params
        self.now = datetime.now(UTC).replace(microsecond=0)
        self.rng = random.Random(42)
        REDACT.register(self.user, "<user>")
        self.sensors = [
            f"{asset}_{measure}"
            for asset in _SAMPLE_ASSETS[params.industry]
            for measure in _SAMPLE_MEASURES
        ]
        self.rate_per_min = int(len(self.sensors) * 60 * _SAMPLE_SENSORS_PER_SECOND)
        self.hours = params.lookback_hours
        self._coverage = 0.995  # fraction of expected rows present in the window

    # ── discovery stubs ────────────────────────────────────────────────────────
    def find_project(self, explicit: str, prefix: str, branch: str = "") -> str:
        return explicit or f"{prefix}-demo"

    def locate(self, target: str, project: str, branch: str, endpoint_ids: tuple[str, ...]) -> str:
        REDACT.register(self._HOT_HOST, "<host>")
        return project

    def use_warehouse(self, warehouse_id: str) -> None:
        return None

    def warehouse_states(self) -> dict[str, str]:
        return {self._WAREHOUSE_ID: "RUNNING"}

    def close(self) -> None:
        return None

    # ── jobs ───────────────────────────────────────────────────────────────────
    def job(self, job_id: int) -> dict[str, Any]:
        mode = self.params.mode
        spark = {RTM_CONF_KEY: "true"} if mode == "realtime" else {}
        classic = {
            "key": "consumer_cluster",
            "spark_version": "18.2.x-scala2.13",
            "node_type": "i3.xlarge",
            "workers": 2,
            "spot": "ON_DEMAND",
            "instance_profile": "arn:aws:iam::123456789012:instance-profile/iot-streaming",
            "runtime_engine": "STANDARD",
            "spark_conf": spark,
        }
        return {
            "name": f"iot-{self.params.industry}-{mode}",
            "is_continuous": mode != "zerobus",
            "continuous": "UNPAUSED" if mode != "zerobus" else None,
            "serverless": mode == "zerobus",
            "tasks": ["producer", "rtm_consumer" if mode == "realtime" else "microbatch_consumer"]
            if mode != "zerobus"
            else ["zerobus_producer"],
            "clusters": [] if mode == "zerobus" else [classic],
        }

    def runs(self, job_id: int, limit: int) -> list[dict[str, Any]]:
        out = []
        start = self.now - timedelta(hours=52)
        for i in range(limit):
            failed = i == 6
            begin = start - timedelta(hours=i * 9)
            end = None if i == 0 else begin + timedelta(hours=9 if not failed else 1.2)
            out.append(
                {
                    "run_id": 5_100_000_000 + (limit - i) * 137,
                    "life_cycle": "RUNNING" if i == 0 else "TERMINATED",
                    "result": None if i == 0 else ("FAILED" if failed else "CANCELED"),
                    "message": "Task consumer failed: stream stopped by Kafka timeout"
                    if failed
                    else None,
                    "start": begin,
                    "end": end,
                    "duration_s": None if i == 0 else (end - begin).total_seconds(),
                }
            )
        return out

    def run_tasks(self, run_id: int) -> list[dict[str, Any]]:
        return [
            {"task": t, "life_cycle": "RUNNING", "result": None, "message": None}
            for t in self.job(self._JOB_ID)["tasks"]
        ]

    def jobs_named(self, name: str) -> list[dict[str, Any]]:
        known = ("Partition Manager", "Rollup Refresh", "Tiering")
        if any(k in name for k in known) or "Retention" in name:
            return [{"job_id": 700_000_000_000 + len(name), "name": name}]
        return []

    # ── Lakebase / Postgres queries ────────────────────────────────────────────
    def _minute_series(self, minutes: int) -> list[dict[str, Any]]:
        base = self.now.replace(second=0) - timedelta(minutes=minutes)
        out = []
        for i in range(minutes):
            dip = 0.55 if i == minutes - 23 else 1.0
            n = int(self.rate_per_min * dip * self.rng.uniform(0.985, 1.015))
            out.append({"bucket": base + timedelta(minutes=i), "n": n})
        return out

    def _hour_series(self) -> list[dict[str, Any]]:
        base = self.now.replace(minute=0, second=0) - timedelta(hours=self.hours)
        return [
            {
                "bucket": base + timedelta(hours=i),
                "n": int(self.rate_per_min * 60 * self.rng.uniform(0.97, 1.0)),
            }
            for i in range(self.hours + 1)
        ]

    def query(self, name: str, **params: Any) -> list[dict[str, Any]]:
        handler = getattr(self, f"_q_{name}", None)
        if handler is None:
            raise CheckSkipped(f"NOT FOUND: sample dataset has no data for {name}")
        return handler(**params)

    def _window_rows(self) -> int:
        return int(self.rate_per_min * 60 * self.hours * self._coverage)

    def _q_cp_setup_by_id(self, **_: Any) -> list[dict[str, Any]]:
        return self._q_cp_setup_by_industry()

    def _q_cp_setup_by_industry(self, **_: Any) -> list[dict[str, Any]]:
        return [
            {
                "id": self._SETUP_ID,
                "industry": self.params.industry,
                "flavor": "kafka-msk",
                "source_type": "kafka",
                "status": "deployed",
                "enabled_modes": ["realtime", "microbatch", "zerobus"],
                "warehouse_id": self._WAREHOUSE_ID,
                "cluster_spec": {"num_workers": 2, "node_type_id": "i3.xlarge"},
                "validation_state": {"realtime": {"passed": True}, "microbatch": {"passed": True}},
                "created_at": self.now - timedelta(days=9),
                "updated_at": self.now - timedelta(days=2, hours=3),
            }
        ]

    def _q_cp_deployment(self, **_: Any) -> list[dict[str, Any]]:
        industry, mode = self.params.industry, self.params.mode
        table = (
            f"acme_catalog.iot_lakehouse.lb__shadow_{industry}_sensor_readings_history"
            if mode == "realtime"
            else f"acme_catalog.iot_lakehouse.{industry}_{mode}"
        )
        return [
            {
                "id": self._DEPLOYMENT_ID,
                "setup_id": self._SETUP_ID,
                "mode": mode,
                "status": "deployed",
                "partition_count": 6,
                "cdf_enabled": mode == "realtime",
                "lakebase_branch": industry if mode == "realtime" else None,
                "lakebase_endpoints": {"primary": "primary", "replica": "replica"},
                "delta_table": table,
                "job_id": str(self._JOB_ID),
                "notebook_path": f"/Users/{self.user}/iot-digital-twin/{industry}/{mode}",
                "created_at": self.now - timedelta(days=9),
                "updated_at": self.now - timedelta(days=2, hours=3),
            }
        ]

    def _q_cp_resources(self, **_: Any) -> list[dict[str, Any]]:
        industry = self.params.industry
        rows = [
            ("uc_schema", "acme_catalog.iot_lakehouse", None, 10),
            ("uc_table", f"acme_catalog.iot_lakehouse.{industry}_metrics", None, 20),
            ("uc_table", f"acme_catalog.iot_lakehouse.{industry}_microbatch", None, 20),
            ("uc_table", f"acme_catalog.iot_lakehouse.{industry}_zerobus", None, 20),
            ("lakebase_branch", industry, "projects/iot-pipelines-demo", 20),
            ("job", str(self._JOB_ID), None, 50),
            ("cdf_sync", f"{industry}-cdf", None, 40),
            ("secret_scope", "iot-streaming-scope", None, 5),
        ]
        return [
            {"resource_type": t, "identifier": i, "parent_ref": p, "teardown_order": o,
             "status": "active", "created_at": self.now - timedelta(days=9)}
            for t, i, p, o in rows
        ]  # fmt: skip

    def _q_cp_alert_events(self, **_: Any) -> list[dict[str, Any]]:
        return [{"events": 31, "opened": 12, "last_event_at": self.now - timedelta(minutes=17)}]

    def _q_lb_total_est(self, **_: Any) -> list[dict[str, Any]]:
        return [
            {"n_estimate": 9_400_000 + self._window_rows(), "min_t": self.now - timedelta(days=13)}
        ]

    def _q_lb_est(self, **_: Any) -> list[dict[str, Any]]:
        return [
            {
                "n_estimate": int(self._window_rows() * 0.98),
                "min_t": self.now - timedelta(hours=self.hours),
            }
        ]

    def _q_lb_latest(self, **_: Any) -> list[dict[str, Any]]:
        return [{"max_t": self.now - timedelta(seconds=3), "lag_s": 3.2, "db_now": self.now}]

    def _q_lb_est_hour(self, **_: Any) -> list[dict[str, Any]]:
        return self._hour_series()

    def _q_lb_rate_minute(self, **_: Any) -> list[dict[str, Any]]:
        return self._minute_series(60)

    def _q_lb_per_sensor(self, **_: Any) -> list[dict[str, Any]]:
        per = self._window_rows() // len(self.sensors)
        return [
            {
                "sensor_name": s,
                "sensor_type": s.split("_")[-1],
                "unit": "unit",
                "n": per - i * 11,
                "first_event_time": self.now - timedelta(hours=self.hours, seconds=-4),
                "last_event_time": self.now - timedelta(seconds=780 if i == 4 else 2 + i % 3),
                "silent_s": 780.0 if i == 4 else float(2 + i % 3),
            }
            for i, s in enumerate(self.sensors)
        ]

    def _q_lb_nulls(self, **_: Any) -> list[dict[str, Any]]:
        row = {"n": self._window_rows()}
        row.update({f"{c}__nulls": 0 for c, *_ in EXPECTED_SENSOR_COLUMNS})
        row["sensor_type__nulls"] = 12
        row["min_value__nulls"] = 0
        return [row]

    def _q_lb_dupes(self, **_: Any) -> list[dict[str, Any]]:
        return [{"n": self._window_rows() // 4, "dup_groups": 24, "dup_rows": 24}]

    def _q_lb_order(self, **_: Any) -> list[dict[str, Any]]:
        return [{"n": ORDER_SAMPLE_ROWS, "out_of_order": 410}]

    def _q_lb_range(self, **_: Any) -> list[dict[str, Any]]:
        return [{"bounded": self._window_rows(), "out_of_range": int(self._window_rows() * 0.003)}]

    def _q_lb_roc(self, **_: Any) -> list[dict[str, Any]]:
        return [{"n": 130_000, "outliers": 158, "max_abs_rate": 41.7}]

    def _q_lb_size(self, **_: Any) -> list[dict[str, Any]]:
        return [{"heap_bytes": 36_507_222_016, "chunks": 120}]

    def _q_lb_columns(self, **_: Any) -> list[dict[str, Any]]:
        pg = {c: t for c, t, *_ in EXPECTED_SENSOR_COLUMNS}
        cols = [("id", "bigint")] + list(pg.items()) + [("source", "text")]
        return [{"column_name": c, "data_type": t, "is_nullable": "YES"} for c, t in cols]

    def _q_lk_version(self, **_: Any) -> list[dict[str, Any]]:
        return [{"version": "0.1.0", "installed_at": self.now - timedelta(days=9)}]

    def _q_lk_registry(self, **_: Any) -> list[dict[str, Any]]:
        return [
            {
                "chunk_interval": "00:15:00",
                "tiering_enabled": True,
                "retention_interval": "14 days",
                "shadow_table_name": f"_shadow_{self.params.industry}_sensor_readings",
                "sync_enabled": True,
            }
        ]

    def _q_lk_chunks(self, **_: Any) -> list[dict[str, Any]]:
        t0 = self.now - timedelta(days=13)
        return [
            {"status": "active", "chunks": 120, "rows": 1_540_000, "bytes": 42_000_000_000,
             "last_tiered_at": None, "oldest": t0, "newest": self.now + timedelta(minutes=45),
             "forward": 3, "created_in_window": 96},
            {"status": "tiered", "chunks": 340, "rows": 6_100_000, "bytes": 0,
             "last_tiered_at": self.now - timedelta(minutes=25), "oldest": t0 - timedelta(days=3),
             "newest": t0, "forward": 0, "created_in_window": 0},
            {"status": "dropped", "chunks": 210, "rows": 3_300_000, "bytes": 0,
             "last_tiered_at": self.now - timedelta(minutes=25), "oldest": t0 - timedelta(days=9),
             "newest": t0 - timedelta(days=3), "forward": 0, "created_in_window": 0},
        ]  # fmt: skip

    def _q_lk_policies(self, **_: Any) -> list[dict[str, Any]]:
        return [
            {"policy_type": "tiered_retention", "enabled": True,
             "config": {"tier_after": "7 days", "drop_after": "14 days"},
             "last_run_at": self.now - timedelta(minutes=25)}
        ]  # fmt: skip

    def _q_lk_tiering(self, **_: Any) -> list[dict[str, Any]]:
        return [
            {"active": 120, "age_eligible": 2, "tiered": 340, "dropped": 210,
             "policy_type": "tiered_retention", "enabled": True,
             "last_run_at": self.now - timedelta(minutes=25), "last_run_age_s": 1500.0}
        ]  # fmt: skip

    def _q_lk_rollups(self, **_: Any) -> list[dict[str, Any]]:
        return [
            {"name": "sensor_1m", "bucket": "00:01:00", "watermark": self.now - timedelta(minutes=2),
             "last_refreshed_at": self.now - timedelta(seconds=95), "refresh_age_s": 95.0,
             "pending_invalidations": 12}
        ]  # fmt: skip

    def _q_lk_lvc(self, **_: Any) -> list[dict[str, Any]]:
        return [{"cached_series": len(self.sensors), "lag_s": 1.4, "update_age_s": 4.0}]

    def _q_lk_gaps(self, **_: Any) -> list[dict[str, Any]]:
        expected = len(self.sensors) * 30
        return [{"sensor_count": len(self.sensors), "observed_buckets": expected - 41}]

    def _q_lk_cdf(self, **_: Any) -> list[dict[str, Any]]:
        return [{"sync_enabled": True, "shadow_table_name": f"_shadow_{self.params.industry}_sensor_readings",
                 "committed_lsn": "0/5A3F9C10", "wal_lag_bytes": 3_250_000}]  # fmt: skip

    def _q_lk_wal2delta(self, **_: Any) -> list[dict[str, Any]]:
        return [{"table_name": f"public._shadow_{self.params.industry}_sensor_readings",
                 "status": "STREAMING", "committed_lsn": "0/5A3F9C10"}]  # fmt: skip

    def _q_lb_recon(self, **_: Any) -> list[dict[str, Any]]:
        return [{"n": self._recon_rows()}]

    def _recon_rows(self) -> int:
        settle = THRESHOLDS.recon_settle_min
        return int(self.rate_per_min * (self.hours * 60 - settle) * self._coverage)

    # ── Unity Catalog ──────────────────────────────────────────────────────────
    def uc(self, name: str, **fields: Any) -> list[dict[str, Any]]:
        handler = getattr(self, f"_uc_{name}", None)
        if handler is None:
            raise CheckSkipped(f"NOT FOUND: sample dataset has no data for {name}")
        return handler(**fields)

    def _uc_uc_tables(self, **f: Any) -> list[dict[str, Any]]:
        industry = f["industry"]
        names = [
            f["history"],
            f"{industry}_metrics",
            f"{industry}_microbatch",
            f"{industry}_zerobus",
        ]
        return [
            {"table_name": n, "table_type": "MANAGED", "data_source_format": "DELTA",
             "last_altered": self.now - timedelta(minutes=2 + i)}
            for i, n in enumerate(names)
        ]  # fmt: skip

    def _uc_uc_columns(self, **f: Any) -> list[dict[str, Any]]:
        if str(f["table_name"]).endswith("_metrics"):
            cols = [(c, "DOUBLE") for c in METRICS_COLUMNS]
        else:
            cols = [(c, d.upper()) for c, _p, d, _n in EXPECTED_SENSOR_COLUMNS]
            if self.params.mode == "microbatch":
                cols.append(("kafka_offset", "BIGINT"))  # an extra column -> schema-drift WARN
        return [{"column_name": c, "data_type": t} for c, t in cols]

    def _uc_uc_total(self, **_: Any) -> list[dict[str, Any]]:
        return [
            {
                "n": 8_900_000 + self._window_rows(),
                "max_t": self.now - timedelta(seconds=self._cold_lag()),
            }
        ]

    def _cold_lag(self) -> int:
        return {"realtime": 430, "microbatch": 240, "zerobus": 35}[self.params.mode]

    def _uc_uc_window(self, **_: Any) -> list[dict[str, Any]]:
        n = int(self._window_rows() * (0.94 if self.params.mode == "realtime" else 0.999))
        return [{"n": n, "min_t": self.now - timedelta(hours=self.hours),
                 "max_t": self.now - timedelta(seconds=self._cold_lag()),
                 "sensors": len(self.sensors), "lag_s": float(self._cold_lag())}]  # fmt: skip

    def _uc_uc_rate_hour(self, **_: Any) -> list[dict[str, Any]]:
        return [dict(r, n=int(r["n"] * 0.94)) for r in self._hour_series()]

    def _uc_uc_nulls(self, **_: Any) -> list[dict[str, Any]]:
        row = {"n": self._window_rows()}
        row.update({f"{c}__nulls": 0 for c, *_ in EXPECTED_SENSOR_COLUMNS})
        return [row]

    def _uc_uc_dupes(self, **_: Any) -> list[dict[str, Any]]:
        return [{"n": self._window_rows() // 4, "dup_groups": 0, "dup_rows": 0}]

    def _uc_uc_recon(self, **_: Any) -> list[dict[str, Any]]:
        n = self._recon_rows()
        return [{"n": int(n * 0.938) if self.params.mode == "realtime" else n}]

    def _uc_uc_history(self, **_: Any) -> list[dict[str, Any]]:
        ops = ["STREAMING UPDATE", "STREAMING UPDATE", "OPTIMIZE", "STREAMING UPDATE"]
        return [
            {"version": 412 - i, "timestamp": self.now - timedelta(seconds=85 + i * 300),
             "operation": ops[i % 4], "numFiles": 2 + i % 3, "numOutputRows": 1500 + i * 20}
            for i in range(15)
        ]  # fmt: skip

    def _uc_uc_detail(self, **_: Any) -> list[dict[str, Any]]:
        return [{"numFiles": 1840, "sizeInBytes": 9_200_000_000, "format": "delta",
                 "partitionColumns": [], "lastModified": self.now - timedelta(seconds=85)}]  # fmt: skip

    def _uc_uc_metrics(self, **f: Any) -> list[dict[str, Any]]:
        step_s = 300 if f["mode"] == "realtime" else 600
        batches = min(self.hours * 3600 // step_s, 1000)
        rng = random.Random(7)
        rows = []
        for i in range(batches):
            ts = self.now - timedelta(seconds=step_s * i + 20)
            spike = 4.5 if i == 37 else 1.0
            rows.append(
                {"mode": f["mode"], "batch_id": 90_000 - i, "timestamp": ts,
                 "num_input_rows": int(step_s * 18 * rng.uniform(0.97, 1.03)),
                 "input_rps": 18.0 * rng.uniform(0.95, 1.05),
                 "processed_rps": 18.4 * rng.uniform(0.95, 1.05),
                 "trigger_ms": 2100 * spike * rng.uniform(0.9, 1.1) * (3 if f["mode"] == "microbatch" else 1),
                 "e2e_p50": 260 * rng.uniform(0.9, 1.15), "e2e_p99": 1450 * spike * rng.uniform(0.9, 1.2),
                 "processing_p50": 120 * rng.uniform(0.9, 1.1), "processing_p99": 610 * rng.uniform(0.9, 1.2),
                 "queuing_p50": 20.0, "queuing_p99": 90.0}
            )  # fmt: skip
        return rows

    def _uc_uc_metrics_progress(self, **f: Any) -> list[dict[str, Any]]:
        progress = {
            "name": f"{self.params.industry}_{f['mode']}",
            "numInputRows": 5400,
            "processedRowsPerSecond": 18.4,
            "durationMs": {"triggerExecution": 2100, "latestOffset": 40},
            "sources": [
                {"description": "KafkaV2[Subscribe[iot-sensor-readings]]", "numInputRows": 5400}
            ],
            "stateOperators": [],
        }
        return [{"timestamp": self.now - timedelta(seconds=20), "progress": json.dumps(progress)}]

    def _uc_uc_zerobus_summary(self, **_: Any) -> list[dict[str, Any]]:
        records = self.rate_per_min * 60 * self.hours
        return [{"last_commit_time": self.now - timedelta(seconds=35), "committed_records": records,
                 "committed_bytes": records * 180, "error_count": 0, "seconds_since_commit": 35.0}]  # fmt: skip

    def _uc_uc_zerobus_series(self, **_: Any) -> list[dict[str, Any]]:
        return [
            {"bucket": r["bucket"], "records": r["n"], "errors": 0} for r in self._hour_series()
        ]


def make_backend(params: Params) -> Any:
    """Return the sample backend or a live workspace backend, per the ``sample`` widget."""
    return SampleBackend(params) if params.sample else LiveBackend(params)

In [10]:
from dataclasses import dataclass, field


class NullBackend:
    """Used when the workspace cannot be reached at all: every access degrades to SKIPPED."""

    label = "no workspace access"
    user = ""
    host = ""

    def __init__(self, reason: str) -> None:
        self.reason = reason

    def __getattr__(self, name: str) -> Any:
        def _unavailable(*_a: Any, **_k: Any) -> Any:
            raise CheckSkipped(f"NO ACCESS: {self.reason}")

        return _unavailable

    def close(self) -> None:
        return None


class Data:
    """Cached, source-tagged access to the backend (one query is only ever run once)."""

    def __init__(self, backend: Any) -> None:
        self.backend = backend
        self._cache: dict[tuple, Any] = {}
        self.sources: dict[str, str] = {}

    def _mark(self, source: str, ok: bool, exc: BaseException | None = None) -> None:
        if ok:
            self.sources[source] = "reached"
        elif self.sources.get(source) != "reached":
            self.sources[source] = classify_error(exc) if exc else "ERROR"

    def _cached(self, key: tuple, source: str, fn: Callable[[], Any]) -> Any:
        if key not in self._cache:
            try:
                self._cache[key] = ("ok", fn())
                self._mark(source, True)
            except Exception as exc:
                self._cache[key] = ("err", exc)
                self._mark(source, False, exc)
        kind, payload = self._cache[key]
        if kind == "err":
            raise payload
        return payload

    def lb(self, name: str, **params: Any) -> list[dict[str, Any]]:
        target = LAKEBASE_SQL[name][0]
        source = "control-plane Lakebase" if target == "app" else "pipeline Lakebase"
        key = ("lb", name, tuple(sorted(params.items())))
        return self._cached(key, source, lambda: self.backend.query(name, **params))

    def lb_one(self, name: str, **params: Any) -> dict[str, Any]:
        rows = self.lb(name, **params)
        if not rows:
            raise CheckSkipped(f"NOT FOUND: {name} returned no rows")
        return rows[0]

    def uc(self, name: str, **fields: Any) -> list[dict[str, Any]]:
        source = "system tables" if "zerobus" in name else "Unity Catalog SQL"
        key = ("uc", name, tuple(sorted(fields.items())))
        return self._cached(key, source, lambda: self.backend.uc(name, **fields))

    def uc_one(self, name: str, **fields: Any) -> dict[str, Any]:
        rows = self.uc(name, **fields)
        if not rows:
            raise CheckSkipped(f"NOT FOUND: {name} returned no rows")
        return rows[0]

    def jobs(self, key: str, fn: Callable[[], Any]) -> Any:
        return self._cached(("jobs", key), "Jobs API", fn)


@dataclass
class Context:
    """Everything resolved about the one pipeline under inspection."""

    setup: dict[str, Any] = field(default_factory=dict)
    deployment: dict[str, Any] = field(default_factory=dict)
    resources: list[dict[str, Any]] = field(default_factory=list)
    job_id: int | None = None
    hot_project: str = ""
    hot_branch: str = ""
    catalog: str = ""
    schema: str = ""
    cold_table: str = ""
    history_table: str = ""
    metrics_table: str = ""
    hot_ready: bool = False
    notes: list[str] = field(default_factory=list)

    @property
    def has_hot_tier(self) -> bool:
        return PARAMS.mode == "realtime"

    @property
    def can_query_uc(self) -> bool:
        return bool(self.catalog and self.schema and self.cold_table)

    def uc_table(self, table: str | None = None) -> str:
        return uc_fqn(self.catalog, self.schema, table or self.cold_table)


CTX = Context()
RUN_STARTED = datetime.now(UTC)

In [11]:
REDACT.enabled = PARAMS.redact
install_output_guards()
SCORECARD = LiveBlock()
HEADER = LiveBlock()
SCORECARD.show(
    heading("Scorecard", 2) + sample_banner() + note("Generated at the end of the run.", "INFO")
)

section,check,status,value,threshold,explanation
7. Cross-checks & data quality,xc_reconciliation,FAIL,"Lakebase 1,531,305 vs Delta 1,436,364 (6.20%)","WARN >= 1.0%, FAIL >= 5.0%","Same closed 24h window [2026-11-19 09:02:09, 2026-11-20 09:02:09) UTC; newest 15 min excluded so CDF flush latency does not count as loss. Rows tiered/dropped from Lakebase in the window would also show up here."
2. Pipeline & job health,job_recent_failures,WARN,1 of last 10,"WARN >= 1, FAIL >= 3",1 failed run(s) among the last 10 (cancelled runs are not counted).
3. Lakebase hot tier,lb_silent_sensors,WARN,1 of 18 (5.6%),"WARN >= 5.0%, FAIL >= 25.0%",Sensors seen in the window but with no reading for 10+ minutes: press_line_1_speed
4. LakeTS outcomes,lakets_gap_fill,WARN,41 of 540 (7.6%),"missing WARN >= 5.0%, FAIL >= 25.0%",Missing 1-minute buckets per sensor over the last 30 minutes (18 sensors).
5. CDF / Unity Catalog cold tier,uc_cdf_lag,WARN,7.1m,"WARN > 300s, FAIL > 1800s",Newest event_time in Lakebase minus newest event_time in Delta (CDF replication lag).
5. CDF / Unity Catalog cold tier,uc_freshness,WARN,7.2m,"WARN > 300s, FAIL > 1800s",Newest event_time in the cold-tier table is 7.2m old.
5. CDF / Unity Catalog cold tier,uc_small_files,WARN,"1,840 files, avg 4.8 MB",WARN: >= 50 files and avg < 16.0 MB,Total 8.6 GB.
6. Streaming metrics,zerobus_ingest,SKIPPED,-,-,NOT APPLICABLE: zerobus ingest system table only applies to the zerobus mode
1. Header & resolved pipeline,control_plane_record,PASS,deployed,setup + deployment record present,"Resolved manufacturing/realtime: deployment status 'deployed', 8 ledger resources."
2. Pipeline & job health,cdf_enabled,PASS,True,cdf_enabled = true,Managed CDF (Lakebase -> Delta) is confirmed in the control plane.


In [12]:
import platform

SECTION_HEADER = "1. Header & resolved pipeline"


def build_backend() -> Any:
    try:
        return make_backend(PARAMS)
    except Exception as exc:
        return NullBackend(f"{classify_error(exc)} while connecting: {REDACT.exception(exc)[:200]}")


BACKEND = build_backend()
DATA = Data(BACKEND)
SECTION_JOBS, SECTION_LB = "2. Pipeline & job health", "3. Lakebase hot tier"
SECTION_LAKETS, SECTION_COLD = "4. LakeTS outcomes", "5. CDF / Unity Catalog cold tier"
SECTION_METRICS, SECTION_CROSS = "6. Streaming metrics", "7. Cross-checks & data quality"


def _register_ids() -> None:
    """Teach the redactor every identifier we resolved so it is masked everywhere."""
    REDACT.register(CTX.setup.get("id"), "<setup-id>")
    REDACT.register(CTX.deployment.get("id"), "<deployment-id>")
    REDACT.register(CTX.setup.get("warehouse_id") or PARAMS.warehouse_id, "<warehouse-id>")
    REDACT.register(PARAMS.setup_id, "<setup-id>")
    REDACT.register(CTX.job_id, "<job-id>")
    REDACT.register(CTX.catalog, "<catalog>")
    REDACT.register(CTX.schema, "<schema>")
    REDACT.register(CTX.deployment.get("notebook_path"), "<notebook-path>")
    for row in CTX.resources:
        if row.get("resource_type") in {"secret_scope", "lakebase_project", "service_principal"}:
            REDACT.register(row.get("identifier"), f"<{row['resource_type']}>")


def _ledger_uc_parts() -> tuple[str, str]:
    """(catalog, schema) from the resource ledger's UC rows."""
    prefix = f"{PARAMS.industry}_"
    for row in CTX.resources:
        parts = str(row.get("identifier") or "").split(".")
        if (
            row.get("resource_type") == "uc_table"
            and len(parts) == 3
            and parts[2].startswith(prefix)
        ):
            return parts[0], parts[1]
    for row in CTX.resources:
        parts = str(row.get("identifier") or "").split(".")
        if row.get("resource_type") == "uc_schema" and len(parts) == 2:
            return parts[0], parts[1]
    return "", ""


def derive_uc_context() -> None:
    """Catalog/schema/table names for this pipeline (widgets > deployment record > ledger)."""
    industry, mode = PARAMS.industry, PARAMS.mode
    parts = str(CTX.deployment.get("delta_table") or "").split(".")
    recorded = parts if len(parts) == 3 and all(parts) else None
    catalog, schema = (recorded[0], recorded[1]) if recorded else _ledger_uc_parts()
    CTX.catalog = PARAMS.uc_catalog or catalog
    CTX.schema = PARAMS.uc_schema or schema
    CTX.history_table = f"lb__shadow_{industry}_sensor_readings_history"
    CTX.metrics_table = f"{industry}_metrics"
    default_cold = CTX.history_table if mode == "realtime" else f"{industry}_{mode}"
    CTX.cold_table = recorded[2] if recorded and recorded[2] else default_cold
    warehouse = PARAMS.warehouse_id or str(CTX.setup.get("warehouse_id") or "")
    if warehouse:
        BACKEND.use_warehouse(warehouse)


def resolve_pipeline() -> Outcome:
    """Find the setup + deployment of ``industry``/``mode`` in the app's control plane."""
    project = BACKEND.find_project(
        PARAMS.app_backend_project, APP_BACKEND_PROJECT_PREFIX, APP_BACKEND_BRANCH
    )
    BACKEND.locate("app", project, APP_BACKEND_BRANCH, (PRIMARY_ENDPOINT,))
    if PARAMS.setup_id:
        setup = DATA.lb_one("cp_setup_by_id", setup_id=PARAMS.setup_id)
    else:
        setup = DATA.lb_one("cp_setup_by_industry", industry=PARAMS.industry)
    CTX.setup = setup
    rows = DATA.lb("cp_deployment", setup_id=setup["id"], mode=PARAMS.mode)
    CTX.resources = DATA.lb("cp_resources", setup_id=setup["id"])
    if not rows:
        raise CheckSkipped(
            f"NOT FOUND: setup exists (status {setup.get('status')}) but has no {PARAMS.mode} deployment"
        )
    CTX.deployment = rows[0]
    job_id = str(CTX.deployment.get("job_id") or "")
    CTX.job_id = int(job_id) if job_id.isdigit() else None
    derive_uc_context()
    _register_ids()
    return Outcome(
        PASS,
        f"{CTX.deployment.get('status')}",
        "setup + deployment record present",
        f"Resolved {PARAMS.industry}/{PARAMS.mode}: deployment status "
        f"'{CTX.deployment.get('status')}', {len(CTX.resources)} ledger resources.",
    )


def environment_rows() -> list[tuple[str, Any]]:
    return [
        ("data backend", BACKEND.label),
        ("python", f"{platform.python_version()} ({platform.system()})"),
        ("databricks runtime", os.environ.get("DATABRICKS_RUNTIME_VERSION", "not on Databricks")),
        ("workspace host", getattr(BACKEND, "host", "") or "-"),
        ("running as", getattr(BACKEND, "user", "") or "-"),
    ]


def render_header(final: bool) -> str:
    deployment, setup = CTX.deployment, CTX.setup
    params_rows = [
        ("industry", PARAMS.industry),
        ("mode", PARAMS.mode),
        ("setup_id (input)", PARAMS.setup_id or "(latest for industry+mode)"),
        ("lookback_hours", PARAMS.lookback_hours),
        ("redact", PARAMS.redact),
        ("sample", PARAMS.sample),
    ]
    pipeline_rows = [
        ("setup id", setup.get("id", "-")),
        ("deployment id", deployment.get("id", "-")),
        ("mode", deployment.get("mode", PARAMS.mode)),
        ("deployment status", deployment.get("status", "-")),
        ("setup status", setup.get("status", "-")),
        ("created", fmt_ts(deployment.get("created_at"))),
        ("updated", fmt_ts(deployment.get("updated_at"))),
        ("cold table", f"{CTX.catalog}.{CTX.schema}.{CTX.cold_table}" if CTX.can_query_uc else "-"),
    ]
    sources = [{"source": k, "state": v} for k, v in sorted(DATA.sources.items())] or [
        {"source": "(none reached yet)", "state": "-"}
    ]
    stamp = f"run timestamp: {fmt_ts(RUN_STARTED)} | redaction: {'ON' if REDACT.enabled else 'OFF'}"
    return (
        heading("1. Header", 2)
        + sample_banner()
        + note(stamp)
        + "".join(note(n, "WARN") for n in PARAM_NOTES)
        + heading("Parameters", 4)
        + kv_table(params_rows)
        + heading("Resolved pipeline", 4)
        + kv_table(pipeline_rows)
        + heading("Environment", 4)
        + kv_table(environment_rows())
        + heading("Data sources reached" + (" (final)" if final else " (so far)"), 4)
        + html_table(sources, ["source", "state"])
    )


REPORT.run(SECTION_HEADER, "control_plane_record", resolve_pipeline)
HEADER.show(render_header(final=False))
finish_section(SECTION_HEADER)

field,value
industry,manufacturing
mode,realtime
setup_id (input),(latest for industry+mode)
lookback_hours,24
redact,True
sample,True
field,value
setup id,<setup-id>
deployment id,<deployment-id>
mode,realtime


check,status,value,threshold,explanation
control_plane_record,PASS,deployed,setup + deployment record present,"Resolved manufacturing/realtime: deployment status 'deployed', 8 ledger resources."


In [13]:
def _need_job() -> int:
    if CTX.job_id is None:
        raise CheckSkipped("NOT FOUND: the deployment record has no job id")
    return CTX.job_id


def _job_info() -> dict[str, Any]:
    return DATA.jobs("job", lambda: BACKEND.job(_need_job()))


def _job_runs() -> list[dict[str, Any]]:
    return DATA.jobs("runs", lambda: BACKEND.runs(_need_job(), JOB_RUN_HISTORY))


def check_deployment_status() -> Outcome:
    status = str(CTX.deployment.get("status") or "unknown")
    if not CTX.deployment:
        raise CheckSkipped("NOT FOUND: no resolved deployment")
    level = PASS if status in HEALTHY_DEPLOYMENT_STATUSES else FAIL if status == "failed" else WARN
    return Outcome(
        level,
        status,
        f"one of {sorted(HEALTHY_DEPLOYMENT_STATUSES)}",
        f"Control-plane deployment status is '{status}' "
        f"(updated {fmt_ts(CTX.deployment.get('updated_at'))}).",
    )


def check_cdf_enabled() -> Outcome:
    if not CTX.has_hot_tier:
        raise NotApplicable("CDF applies to realtime deployments only")
    if not CTX.deployment:
        raise CheckSkipped("NOT FOUND: pipeline was not resolved")
    enabled = bool(CTX.deployment.get("cdf_enabled"))
    return Outcome(
        PASS if enabled else FAIL,
        str(enabled),
        "cdf_enabled = true",
        "Managed CDF (Lakebase -> Delta) is confirmed in the control plane."
        if enabled
        else "CDF is not confirmed, so the realtime cold tier is not being fed.",
    )


def check_last_run() -> Outcome:
    runs = _job_runs()
    if not runs:
        return Outcome(WARN, "no runs", "latest run RUNNING / SUCCESS", "The job has never run.")
    latest = runs[0]
    life, result = latest["life_cycle"], latest["result"]
    continuous = PARAMS.mode != "zerobus"
    if life in ACTIVE_RUN_STATES:
        level, why = PASS, "job run is active"
    elif result in BAD_RESULT_STATES and result != "CANCELED":
        level, why = FAIL, f"last run ended {result}: {latest.get('message') or 'no message'}"
    elif continuous:
        level, why = FAIL, f"continuous job has no active run (last run {life}/{result})"
    elif result == "SUCCESS":
        level, why = PASS, "run-triggered job finished successfully"
    else:
        level, why = WARN, f"last run ended {result or life}"
    return Outcome(level, f"{life}/{result or '-'}", "RUNNING (continuous) / SUCCESS", why)


def check_recent_failures() -> Outcome:
    runs = _job_runs()
    failed = sum(r["result"] in BAD_RESULT_STATES - {"CANCELED"} for r in runs)
    level = status_high(failed, THRESHOLDS.failed_runs_warn, THRESHOLDS.failed_runs_fail)
    return Outcome(
        level,
        f"{failed} of last {len(runs)}",
        f"WARN >= {THRESHOLDS.failed_runs_warn}, FAIL >= {THRESHOLDS.failed_runs_fail}",
        f"{failed} failed run(s) among the last {len(runs)} (cancelled runs are not counted).",
    )


def check_task_health() -> Outcome:
    runs = _job_runs()
    if not runs:
        raise CheckSkipped("NOT FOUND: no runs to inspect")
    tasks = DATA.jobs("tasks", lambda: BACKEND.run_tasks(runs[0]["run_id"]))
    broken = [
        t
        for t in tasks
        if t["life_cycle"] == "INTERNAL_ERROR" or t["result"] in BAD_RESULT_STATES - {"CANCELED"}
    ]
    names = ", ".join(t["task"] for t in broken) or "none"
    return Outcome(
        FAIL if broken else PASS,
        f"{len(broken)} broken of {len(tasks)}",
        "0 broken tasks",
        f"Broken tasks in the latest run: {names}.",
    )


def check_compute_config() -> Outcome:
    job = _job_info()
    mode = PARAMS.mode
    if mode == "zerobus":
        ok = bool(job["serverless"])
        return Outcome(
            PASS if ok else WARN,
            "serverless" if ok else "classic",
            "serverless producer job",
            "Zerobus producer runs on serverless compute." if ok else "Expected a serverless job.",
        )
    clusters = job["clusters"]
    if not clusters:
        return Outcome(
            WARN, "no job cluster", "classic job cluster", "No job cluster found on the job."
        )
    if mode == "realtime":
        enabled = any(c["spark_conf"].get(RTM_CONF_KEY) == "true" for c in clusters)
        return Outcome(
            PASS if enabled else FAIL,
            "RTM on" if enabled else "RTM off",
            f"{RTM_CONF_KEY} = true",
            "Real-time mode is enabled on the job cluster."
            if enabled
            else "Real-time mode is NOT enabled on the job cluster.",
        )
    return Outcome(
        PASS,
        f"{len(clusters)} cluster(s)",
        "classic job cluster",
        "Micro-batch uses a classic job cluster.",
    )


def view_job_config() -> None:
    job = _job_info()
    rows = [
        ("job name", job["name"]),
        ("continuous", job["is_continuous"]),
        ("serverless", job["serverless"]),
        ("tasks", ", ".join(job["tasks"])),
    ]
    emit_html(heading("Job definition", 4) + kv_table(rows))
    if job["clusters"]:
        cluster_rows = [
            {
                "cluster": c["key"],
                "runtime": c["spark_version"],
                "node": c["node_type"],
                "workers": c["workers"],
                "availability": c["spot"],
                "engine": c["runtime_engine"],
                "instance profile": c.get("instance_profile") or "-",
                "RTM conf": c["spark_conf"].get(RTM_CONF_KEY, "-"),
            }
            for c in job["clusters"]
        ]
        emit_html(heading("Job cluster configuration", 4) + html_table(cluster_rows))


def view_run_history() -> None:
    runs = _job_runs()
    rows = [
        {
            "run": f"#{i}",
            "state": f"{r['life_cycle']}/{r['result'] or '-'}",
            "started": fmt_ts(r["start"]),
            "duration": fmt_seconds(r["duration_s"]) if r["duration_s"] else "running",
            "error": (r["message"] or "")[:120],
        }
        for i, r in enumerate(runs)
    ]
    emit_html(heading("Recent runs (newest first)", 4) + html_table(rows))
    bars = [(f"#{i}", (r["duration_s"] or 0) / 3600) for i, r in reversed(list(enumerate(runs)))]
    emit_html(svg_bar_chart(bars, title="Run duration (hours, oldest to newest; active run = 0)"))


def view_streaming_progress() -> None:
    if PARAMS.mode not in SPARK_METRIC_MODES or not CTX.can_query_uc:
        raise NotApplicable("streaming query progress is only recorded for Spark modes")
    table = CTX.uc_table(CTX.metrics_table)
    row = DATA.uc_one("uc_metrics_progress", table=table, mode=PARAMS.mode)
    progress = json.loads(row["progress"]) if isinstance(row["progress"], str) else row["progress"]
    source = (progress.get("sources") or [{}])[0]
    rows = [
        ("reported", fmt_ts(row["timestamp"])),
        ("query name", progress.get("name")),
        ("input rows (batch)", progress.get("numInputRows")),
        ("processed rows/s", progress.get("processedRowsPerSecond")),
        ("trigger execution ms", (progress.get("durationMs") or {}).get("triggerExecution")),
        ("source", source.get("description")),
    ]
    emit_html(heading("Latest streaming query progress", 4) + kv_table(rows))


emit_html(heading("2. Pipeline & job health", 2) + sample_banner())
REPORT.run(SECTION_JOBS, "deployment_status", check_deployment_status)
REPORT.run(SECTION_JOBS, "cdf_enabled", check_cdf_enabled)
REPORT.run(SECTION_JOBS, "job_last_run", check_last_run)
REPORT.run(SECTION_JOBS, "job_recent_failures", check_recent_failures)
REPORT.run(SECTION_JOBS, "job_task_health", check_task_health)
REPORT.run(SECTION_JOBS, "job_compute", check_compute_config)
REPORT.view("job definition", view_job_config)
REPORT.view("run history", view_run_history)
REPORT.view("streaming progress", view_streaming_progress)
finish_section(SECTION_JOBS)

## 2. Pipeline & job health

SAMPLE DATA — synthetic dataset, no workspace was contacted

field,value
job name,iot-manufacturing-realtime
continuous,True
serverless,False
tasks,"producer, rtm_consumer"


cluster,runtime,node,workers,availability,engine,instance profile,RTM conf
consumer_cluster,18.2.x-scala2.13,i3.xlarge,2,ON_DEMAND,STANDARD,<arn>,true


run,state,started,duration,error
#0,RUNNING/-,2026-11-18 05:17:09 UTC,running,
#1,TERMINATED/CANCELED,2026-11-17 20:17:09 UTC,9.0h,
#2,TERMINATED/CANCELED,2026-11-17 11:17:09 UTC,9.0h,
#3,TERMINATED/CANCELED,2026-11-17 02:17:09 UTC,9.0h,
#4,TERMINATED/CANCELED,2026-11-16 17:17:09 UTC,9.0h,
#5,TERMINATED/CANCELED,2026-11-16 08:17:09 UTC,9.0h,
#6,TERMINATED/FAILED,2026-11-15 23:17:09 UTC,72.0m,Task consumer failed: stream stopped by Kafka timeout
#7,TERMINATED/CANCELED,2026-11-15 14:17:09 UTC,9.0h,
#8,TERMINATED/CANCELED,2026-11-15 05:17:09 UTC,9.0h,
#9,TERMINATED/CANCELED,2026-11-14 20:17:09 UTC,9.0h,


field,value
reported,2026-11-20 09:16:49 UTC
query name,manufacturing_realtime
input rows (batch),"5,400"
processed rows/s,18.4
trigger execution ms,"2,100"
source,KafkaV2[Subscribe[iot-sensor-readings]]


check,status,value,threshold,explanation
deployment_status,PASS,deployed,one of ['deployed'],Control-plane deployment status is 'deployed' (updated 2026-11-18 06:17:09 UTC).
cdf_enabled,PASS,True,cdf_enabled = true,Managed CDF (Lakebase -> Delta) is confirmed in the control plane.
job_last_run,PASS,RUNNING/-,RUNNING (continuous) / SUCCESS,job run is active
job_recent_failures,WARN,1 of last 10,"WARN >= 1, FAIL >= 3",1 failed run(s) among the last 10 (cancelled runs are not counted).
job_task_health,PASS,0 broken of 2,0 broken tasks,Broken tasks in the latest run: none.
job_compute,PASS,RTM on,spark.databricks.streaming.realTimeMode.enabled = true,Real-time mode is enabled on the job cluster.


In [14]:
def _require_hot_tier() -> None:
    if not CTX.has_hot_tier:
        raise NotApplicable(f"{PARAMS.mode} writes Delta only - there is no Lakebase hot tier")
    if not CTX.deployment:
        raise CheckSkipped("NOT FOUND: pipeline was not resolved")


def connect_hot_tier() -> Outcome:
    """Resolve the pipeline Lakebase project/branch/endpoints and prove a read-only query works."""
    _require_hot_tier()
    ledger_branch = next(
        (r for r in CTX.resources if r.get("resource_type") == "lakebase_branch"), {}
    )
    parent = str(ledger_branch.get("parent_ref") or "")
    explicit = PARAMS.pipelines_project or (
        parent.split("/", 1)[1] if parent.startswith("projects/") else ""
    )
    CTX.hot_branch = str(CTX.deployment.get("lakebase_branch") or PARAMS.industry)
    CTX.hot_project = BACKEND.find_project(explicit, PIPELINES_PROJECT_PREFIX, CTX.hot_branch)
    BACKEND.locate("data", CTX.hot_project, CTX.hot_branch, (REPLICA_ENDPOINT, PRIMARY_ENDPOINT))
    BACKEND.locate("ctl", CTX.hot_project, CTX.hot_branch, (PRIMARY_ENDPOINT,))
    REDACT.register(CTX.hot_project, "<lakebase-project>")
    columns = DATA.lb("lb_columns")
    CTX.hot_ready = True
    return Outcome(
        PASS if columns else FAIL,
        f"{len(columns)} columns",
        "read-only connection + sensor_readings readable",
        f"Connected read-only (OAuth) to branch '{CTX.hot_branch}'; reads use the replica when present.",
    )


def _hot() -> None:
    _require_hot_tier()
    if not CTX.hot_ready:
        raise CheckSkipped("NO ACCESS: Lakebase connection was not established (see lb_connect)")


def _chunks() -> dict[str, dict[str, Any]]:
    return {r["status"]: r for r in DATA.lb("lk_chunks", hours=PARAMS.lookback_hours)}


def scan_hours() -> int:
    """Hours of the lookback that exact scans may cover (bounded by ``EXACT_SCAN_MAX_ROWS``)."""
    estimate = int(DATA.lb_one("lb_est", hours=PARAMS.lookback_hours)["n_estimate"] or 0)
    recent_hour = sum(int(r["n"]) for r in DATA.lb("lb_rate_minute"))  # measured, not estimated
    per_hour = max(estimate / PARAMS.lookback_hours, recent_hour)
    if per_hour * PARAMS.lookback_hours <= EXACT_SCAN_MAX_ROWS:
        return PARAMS.lookback_hours
    return max(1, min(PARAMS.lookback_hours, int(EXACT_SCAN_MAX_ROWS / per_hour)))


def quality_hours() -> int:
    return min(QUALITY_HOURS, scan_hours())


def hot_window() -> dict[str, Any]:
    """Window facts for the hot tier: exact when affordable, partition-statistics estimate else."""
    hours = scan_hours()
    exact = hours == PARAMS.lookback_hours
    sensors = DATA.lb("lb_per_sensor", hours=hours)
    scan_n = sum(int(s["n"] or 0) for s in sensors)
    first = min((t for t in (to_dt(s["first_event_time"]) for s in sensors) if t), default=None)
    latest = DATA.lb_one("lb_latest")
    estimate = DATA.lb_one("lb_est", hours=PARAMS.lookback_hours)
    return {
        "n": scan_n if exact else estimate["n_estimate"],
        "exact": exact,
        "scan_hours": hours,
        "scan_n": scan_n,
        "sensors": len(sensors),
        "min_t": first if exact else estimate["min_t"],
        "max_t": latest["max_t"],
        "lag_s": latest["lag_s"],
        "db_now": latest["db_now"],
    }


def _scope_note(window: dict[str, Any]) -> str:
    if window["exact"]:
        return ""
    return (
        f" Row count is an estimate from partition statistics; detail scans cover the newest "
        f"{window['scan_hours']}h to stay within the query budget."
    )


def check_chronotable() -> Outcome:
    _hot()
    registry = DATA.lb_one("lk_registry")
    chunks = _chunks()
    forward = int((chunks.get("active") or {}).get("forward") or 0)
    level = PASS if forward >= THRESHOLDS.forward_partitions_min else FAIL
    return Outcome(
        level,
        f"{forward} forward partitions",
        f">= {THRESHOLDS.forward_partitions_min}",
        f"sensor_readings is a registered ChronoTable (chunk interval {registry['chunk_interval']}).",
    )


def check_rows_window() -> Outcome:
    _hot()
    window = hot_window()
    n = int(window["n"] or 0)
    level = PASS if n >= THRESHOLDS.min_rows_in_window else FAIL
    return Outcome(
        level,
        ("~" if not window["exact"] else "") + fmt_num(n),
        f">= {THRESHOLDS.min_rows_in_window} rows in {PARAMS.lookback_hours}h",
        f"{fmt_num(n)} rows in the last {PARAMS.lookback_hours}h "
        f"({fmt_num(window['sensors'])} sensors, {fmt_ts(window['min_t'])} to {fmt_ts(window['max_t'])})."
        + _scope_note(window),
    )


def check_hot_freshness() -> Outcome:
    _hot()
    lag = num(DATA.lb_one("lb_latest")["lag_s"])
    warn, fail = THRESHOLDS.freshness(PARAMS.mode)
    return Outcome(
        status_high(lag, warn, fail),
        fmt_seconds(lag),
        f"WARN > {warn}s, FAIL > {fail}s",
        f"Newest event_time is {fmt_seconds(lag)} old (database clock, no clock skew).",
    )


def _window_vs_recent_rate() -> tuple[float, float]:
    window = hot_window()
    series = DATA.lb("lb_rate_minute")[:-1]  # drop the current, partial minute
    recent = series[-5:]
    recent_rate = sum(int(r["n"]) for r in recent) / max(len(recent), 1)
    average = int(window["scan_n"] or 0) / max(window["scan_hours"] * 60, 1)
    return recent_rate, average


def check_ingest_rate() -> Outcome:
    _hot()
    recent, average = _window_vs_recent_rate()
    ratio = recent / average if average else None
    return Outcome(
        status_low(ratio, THRESHOLDS.rate_drop_warn_ratio, THRESHOLDS.rate_drop_fail_ratio),
        f"{fmt_num(recent, 0)}/min vs {fmt_num(average, 0)}/min",
        f"ratio WARN <= {THRESHOLDS.rate_drop_warn_ratio}, FAIL <= {THRESHOLDS.rate_drop_fail_ratio}",
        "Last 5 complete minutes compared with the window average ingest rate.",
    )


def check_silent_sensors() -> Outcome:
    _hot()
    sensors = DATA.lb("lb_per_sensor", hours=scan_hours())
    silent = [s for s in sensors if (num(s["silent_s"]) or 0) > 600]
    pct = 100 * len(silent) / len(sensors) if sensors else None
    return Outcome(
        status_high(pct, THRESHOLDS.silent_sensor_warn_pct, THRESHOLDS.silent_sensor_fail_pct),
        f"{len(silent)} of {len(sensors)} ({fmt_pct(pct, 1)})",
        f"WARN >= {THRESHOLDS.silent_sensor_warn_pct}%, FAIL >= {THRESHOLDS.silent_sensor_fail_pct}%",
        "Sensors seen in the window but with no reading for 10+ minutes: "
        + (", ".join(s["sensor_name"] for s in silent[:5]) or "none"),
    )


def _null_pcts(row: dict[str, Any]) -> dict[str, float]:
    total = max(int(row["n"] or 0), 1)
    return {c: 100 * int(row[f"{c}__nulls"] or 0) / total for c, *_ in EXPECTED_SENSOR_COLUMNS}


def grade_nulls(pcts: dict[str, float], hours: int) -> Outcome:
    required = {c for c, _p, _d, not_null in EXPECTED_SENSOR_COLUMNS if not_null}
    worst_required = max((pcts[c] for c in required), default=0.0)
    worst_optional = max((v for c, v in pcts.items() if c not in required), default=0.0)
    if worst_required > 0:
        level = FAIL
    else:
        level = status_high(worst_optional, THRESHOLDS.null_pct_warn, THRESHOLDS.null_pct_fail)
    offenders = ", ".join(f"{c} {v:.3f}%" for c, v in pcts.items() if v > 0) or "none"
    return Outcome(
        level,
        f"required {worst_required:.3f}%, optional max {worst_optional:.3f}%",
        f"required = 0%, optional WARN >= {THRESHOLDS.null_pct_warn}%, FAIL >= {THRESHOLDS.null_pct_fail}%",
        f"Columns with NULLs in the newest {hours}h: {offenders}.",
    )


def check_hot_nulls() -> Outcome:
    _hot()
    return grade_nulls(_null_pcts(DATA.lb_one("lb_nulls", qhours=quality_hours())), quality_hours())


def check_hot_duplicates() -> Outcome:
    _hot()
    dupes = DATA.lb_one("lb_dupes", qhours=quality_hours())
    pct = 100 * int(dupes["dup_rows"] or 0) / max(int(dupes["n"] or 0), 1)
    return Outcome(
        status_high(pct, THRESHOLDS.duplicate_pct_warn, THRESHOLDS.duplicate_pct_fail),
        f"{fmt_num(dupes['dup_rows'])} rows ({pct:.4f}%)",
        f"WARN >= {THRESHOLDS.duplicate_pct_warn}%, FAIL >= {THRESHOLDS.duplicate_pct_fail}%",
        f"Repeated (sensor_name, event_time, value) triples in the newest {quality_hours()}h.",
    )


def check_out_of_order() -> Outcome:
    _hot()
    order = DATA.lb_one("lb_order", limit=ORDER_SAMPLE_ROWS)
    pct = 100 * int(order["out_of_order"] or 0) / max(int(order["n"] or 0), 1)
    return Outcome(
        status_high(pct, THRESHOLDS.out_of_order_pct_warn, THRESHOLDS.out_of_order_pct_fail),
        f"{fmt_num(order['out_of_order'])} of {fmt_num(order['n'])} ({pct:.2f}%)",
        f"WARN >= {THRESHOLDS.out_of_order_pct_warn}%, FAIL >= {THRESHOLDS.out_of_order_pct_fail}%",
        "Rows whose event_time is earlier than the previous row (by insertion id) in a sample of the newest hour.",
    )


def check_hot_size() -> Outcome:
    _hot()
    size = DATA.lb_one("lb_size")
    total_gb = (num(size["heap_bytes"]) or 0) / 1024**3
    return Outcome(
        PASS if total_gb < THRESHOLDS.lb_size_gb_warn else WARN,
        f"{total_gb:,.1f} GB",
        f"WARN >= {THRESHOLDS.lb_size_gb_warn} GB",
        f"Heap of {fmt_num(size['chunks'])} active partitions (indexes excluded; from partition statistics).",
    )


def view_hot_summary() -> None:
    _hot()
    window = hot_window()
    total = DATA.lb_one("lb_total_est")
    rows = [
        ("rows total", f"~{fmt_num(total['n_estimate'])} (estimate from partition statistics)"),
        ("oldest active partition starts", fmt_ts(total["min_t"])),
        (
            f"rows in last {PARAMS.lookback_hours}h",
            ("" if window["exact"] else "~")
            + fmt_num(window["n"])
            + ("" if window["exact"] else " (estimate)"),
        ),
        ("event_time range", f"{fmt_ts(window['min_t'])} to {fmt_ts(window['max_t'])}"),
        ("freshness lag", fmt_seconds(window["lag_s"])),
        (f"distinct sensors (newest {window['scan_hours']}h)", fmt_num(window["sensors"])),
    ]
    emit_html(heading("Hot-tier summary", 4) + kv_table(rows))


def _epoch(value: Any) -> float:
    parsed = to_dt(value)
    return parsed.timestamp() if parsed else 0.0


def view_ingest_charts() -> None:
    _hot()
    minutes = DATA.lb("lb_rate_minute")[:-1]
    emit_html(
        svg_line_chart(
            {"rows/min": [(_epoch(r["bucket"]), float(r["n"])) for r in minutes]},
            title="Ingest rate - last 60 minutes",
            y_label="rows per minute",
            x_ends=(fmt_ts(minutes[0]["bucket"]) if minutes else "", "now"),
        )
    )
    hours = DATA.lb("lb_est_hour", hours=PARAMS.lookback_hours)
    emit_html(
        svg_bar_chart(
            [(fmt_ts(r["bucket"])[5:16], float(r["n"])) for r in hours],
            title=f"Ingest volume per hour (partition statistics) - last {PARAMS.lookback_hours}h",
            y_label="rows per hour",
        )
    )


def view_sensor_cardinality() -> None:
    _hot()
    sensors = DATA.lb("lb_per_sensor", hours=scan_hours())
    rows = [
        {"sensor": s["sensor_name"], "type": s["sensor_type"], "rows": s["n"],
         "last event": fmt_ts(s["last_event_time"]), "silent for": fmt_seconds(s["silent_s"])}
        for s in sensors[:12]
    ]  # fmt: skip
    emit_html(heading(f"Per-sensor cardinality (top 12 of {len(sensors)})", 4) + html_table(rows))


emit_html(heading("3. Lakebase hot tier", 2) + sample_banner())
REPORT.run(SECTION_LB, "lb_connect", connect_hot_tier)
REPORT.run(SECTION_LB, "lb_chronotable", check_chronotable)
REPORT.run(SECTION_LB, "lb_rows_window", check_rows_window)
REPORT.run(SECTION_LB, "lb_freshness", check_hot_freshness)
REPORT.run(SECTION_LB, "lb_ingest_rate", check_ingest_rate)
REPORT.run(SECTION_LB, "lb_silent_sensors", check_silent_sensors)
REPORT.run(SECTION_LB, "lb_nulls", check_hot_nulls)
REPORT.run(SECTION_LB, "lb_duplicates", check_hot_duplicates)
REPORT.run(SECTION_LB, "lb_out_of_order", check_out_of_order)
REPORT.run(SECTION_LB, "lb_size", check_hot_size)
REPORT.view("hot-tier summary", view_hot_summary)
REPORT.view("ingest charts", view_ingest_charts)
REPORT.view("sensor cardinality", view_sensor_cardinality)
finish_section(SECTION_LB)

## 3. Lakebase hot tier

SAMPLE DATA — synthetic dataset, no workspace was contacted

field,value
rows total,"~10,947,424 (estimate from partition statistics)"
oldest active partition starts,2026-11-07 09:17:09 UTC
rows in last 24h,"1,545,741"
event_time range,2026-11-19 09:17:13 UTC to 2026-11-20 09:17:06 UTC
freshness lag,3s
distinct sensors (newest 24h),18


sensor,type,rows,last event,silent for
press_line_1_temperature,temperature,"85,968",2026-11-20 09:17:07 UTC,2s
press_line_1_vibration,vibration,"85,957",2026-11-20 09:17:06 UTC,3s
press_line_1_pressure,pressure,"85,946",2026-11-20 09:17:05 UTC,4s
press_line_1_current,current,"85,935",2026-11-20 09:17:07 UTC,2s
press_line_1_speed,speed,"85,924",2026-11-20 09:04:09 UTC,13.0m
press_line_1_humidity,humidity,"85,913",2026-11-20 09:17:05 UTC,4s
cnc_cell_2_temperature,temperature,"85,902",2026-11-20 09:17:07 UTC,2s
cnc_cell_2_vibration,vibration,"85,891",2026-11-20 09:17:06 UTC,3s
cnc_cell_2_pressure,pressure,"85,880",2026-11-20 09:17:05 UTC,4s
cnc_cell_2_current,current,"85,869",2026-11-20 09:17:07 UTC,2s


check,status,value,threshold,explanation
lb_connect,PASS,13 columns,read-only connection + sensor_readings readable,Connected read-only (OAuth) to branch 'manufacturing'; reads use the replica when present.
lb_chronotable,PASS,3 forward partitions,>= 1,sensor_readings is a registered ChronoTable (chunk interval 00:15:00).
lb_rows_window,PASS,"1,545,741",>= 1 rows in 24h,"1,545,741 rows in the last 24h (18 sensors, 2026-11-19 09:17:13 UTC to 2026-11-20 09:17:06 UTC)."
lb_freshness,PASS,3s,"WARN > 120s, FAIL > 600s","Newest event_time is 3s old (database clock, no clock skew)."
lb_ingest_rate,PASS,"1,078/min vs 1,073/min","ratio WARN <= 0.5, FAIL <= 0.1",Last 5 complete minutes compared with the window average ingest rate.
lb_silent_sensors,WARN,1 of 18 (5.6%),"WARN >= 5.0%, FAIL >= 25.0%",Sensors seen in the window but with no reading for 10+ minutes: press_line_1_speed
lb_nulls,PASS,"required 0.000%, optional max 0.001%","required = 0%, optional WARN >= 0.1%, FAIL >= 5.0%",Columns with NULLs in the newest 6h: sensor_type 0.001%.
lb_duplicates,PASS,24 rows (0.0062%),"WARN >= 0.01%, FAIL >= 1.0%","Repeated (sensor_name, event_time, value) triples in the newest 6h."
lb_out_of_order,PASS,"410 of 50,000 (0.82%)","WARN >= 2.0%, FAIL >= 20.0%",Rows whose event_time is earlier than the previous row (by insertion id) in a sample of the newest hour.
lb_size,PASS,34.0 GB,WARN >= 50.0 GB,Heap of 120 active partitions (indexes excluded; from partition statistics).


In [15]:
GAP_WINDOW_MINUTES = 30  # lk_gaps looks at the last 30 one-minute buckets
LAKETS_JOB_KINDS = ("Partition Manager", "Rollup Refresh", "Tiering", "Retention")


def check_lakets_installed() -> Outcome:
    _hot()
    version = DATA.lb_one("lk_version")
    return Outcome(
        PASS, str(version["version"]), "lakets._version present",
        f"LakeTS {version['version']} installed {fmt_ts(version['installed_at'])}.",
    )  # fmt: skip


def check_lakets_partitions() -> Outcome:
    _hot()
    chunks = _chunks()
    active = chunks.get("active") or {}
    forward, created = int(active.get("forward") or 0), int(active.get("created_in_window") or 0)
    return Outcome(
        PASS if forward >= THRESHOLDS.forward_partitions_min else FAIL,
        f"{forward} forward, {created} created in {PARAMS.lookback_hours}h",
        f"forward partitions >= {THRESHOLDS.forward_partitions_min}",
        f"Active {fmt_num(active.get('chunks'))}, tiered {fmt_num((chunks.get('tiered') or {}).get('chunks'))}, "
        f"dropped {fmt_num((chunks.get('dropped') or {}).get('chunks'))} partitions.",
    )


def check_lakets_tiering() -> Outcome:
    _hot()
    rows = DATA.lb("lk_tiering")
    if not rows or rows[0].get("policy_type") is None:
        return Outcome(
            WARN,
            "no policy",
            "tiering/retention policy enabled",
            "No tiering policy is registered.",
        )
    row = rows[0]
    age_h = (num(row.get("last_run_age_s")) or 0) / 3600
    eligible = int(row.get("age_eligible") or 0)
    stale = eligible > 0 and age_h > THRESHOLDS.tiering_idle_h_warn
    level = WARN if (not row.get("enabled") or stale) else PASS
    return Outcome(
        level,
        f"{row['policy_type']}, last run {age_h:.1f}h ago, {eligible} eligible",
        f"eligible chunks with last run > {THRESHOLDS.tiering_idle_h_warn}h = WARN",
        f"Tiered {fmt_num(row['tiered'])}, dropped {fmt_num(row['dropped'])} partitions so far; "
        f"policy {'enabled' if row.get('enabled') else 'DISABLED'}.",
    )


def check_lakets_rollups() -> Outcome:
    _hot()
    rollups = DATA.lb("lk_rollups")
    if not rollups:
        return Outcome(
            WARN, "no rollups", "rollup registered", "No rollup is registered for sensor_readings."
        )
    worst_age = max(num(r["refresh_age_s"]) or 0 for r in rollups)
    pending = sum(int(r["pending_invalidations"] or 0) for r in rollups)
    level = status_high(worst_age, THRESHOLDS.rollup_age_s_warn, THRESHOLDS.rollup_age_s_fail)
    if level == PASS and pending >= THRESHOLDS.rollup_pending_warn:
        level = WARN
    return Outcome(
        level,
        f"refresh age {fmt_seconds(worst_age)}, {pending} pending",
        f"age WARN > {THRESHOLDS.rollup_age_s_warn}s / FAIL > {THRESHOLDS.rollup_age_s_fail}s; "
        f"pending WARN >= {THRESHOLDS.rollup_pending_warn}",
        f"{len(rollups)} rollup(s): " + ", ".join(r["name"] for r in rollups),
    )


def check_lakets_lvc() -> Outcome:
    _hot()
    lvc = DATA.lb_one("lk_lvc")
    lag = num(lvc["lag_s"])
    return Outcome(
        status_high(lag, THRESHOLDS.lvc_lag_s_warn, THRESHOLDS.lvc_lag_s_fail),
        f"lag {fmt_seconds(lag)}, {lvc['cached_series']} series",
        f"WARN > {THRESHOLDS.lvc_lag_s_warn}s, FAIL > {THRESHOLDS.lvc_lag_s_fail}s",
        "Last-value cache event_time versus the source table (feeds the live dashboards).",
    )


def check_gap_fill() -> Outcome:
    _hot()
    gaps = DATA.lb_one("lk_gaps")
    if int(gaps["sensor_count"] or 0) == 0:
        return Outcome(
            FAIL,
            "0 sensors reported",
            "readings in the last 30 minutes",
            "No sensor reported a reading in the last 30 minutes.",
        )
    expected = int(gaps["sensor_count"] or 0) * GAP_WINDOW_MINUTES
    missing = max(expected - int(gaps["observed_buckets"] or 0), 0)
    pct = 100 * missing / expected if expected else None
    return Outcome(
        status_high(pct, THRESHOLDS.gap_missing_pct_warn, THRESHOLDS.gap_missing_pct_fail),
        f"{fmt_num(missing)} of {fmt_num(expected)} ({fmt_pct(pct, 1)})",
        f"missing WARN >= {THRESHOLDS.gap_missing_pct_warn}%, FAIL >= {THRESHOLDS.gap_missing_pct_fail}%",
        f"Missing 1-minute buckets per sensor over the last 30 minutes ({gaps['sensor_count']} sensors).",
    )


def check_rate_of_change() -> Outcome:
    _hot()
    roc_hours = min(scan_hours(), 2)
    roc = DATA.lb_one("lb_roc", roc_hours=roc_hours, sigma=THRESHOLDS.roc_sigma)
    pct = 100 * int(roc["outliers"] or 0) / max(int(roc["n"] or 0), 1)
    return Outcome(
        status_high(pct, THRESHOLDS.roc_outlier_pct_warn, THRESHOLDS.roc_outlier_pct_fail),
        f"{fmt_num(roc['outliers'])} outliers ({pct:.2f}%)",
        f"> {THRESHOLDS.roc_sigma} sigma; WARN >= {THRESHOLDS.roc_outlier_pct_warn}%, "
        f"FAIL >= {THRESHOLDS.roc_outlier_pct_fail}%",
        f"Per-sensor rate of change over {roc_hours}h; max |rate| {fmt_num(roc['max_abs_rate'])}/s.",
    )


def check_value_range() -> Outcome:
    _hot()
    rng = DATA.lb_one("lb_range", qhours=quality_hours())
    bounded = int(rng["bounded"] or 0)
    pct = 100 * int(rng["out_of_range"] or 0) / bounded if bounded else None
    return Outcome(
        status_high(pct, THRESHOLDS.out_of_range_pct_warn, THRESHOLDS.out_of_range_pct_fail),
        f"{fmt_num(rng['out_of_range'])} of {fmt_num(bounded)} ({fmt_pct(pct)})",
        f"WARN >= {THRESHOLDS.out_of_range_pct_warn}%, FAIL >= {THRESHOLDS.out_of_range_pct_fail}%",
        f"Value distribution (newest {quality_hours()}h): readings outside the [min_value, max_value] on the same row.",
    )


def check_lakets_cdf() -> Outcome:
    _hot()
    cdf = DATA.lb_one("lk_cdf")
    wal = DATA.lb("lk_wal2delta", shadow=CDF_SHADOW_TABLE)
    streaming = bool(wal) and str(wal[0]["status"]).upper() == "STREAMING"
    lag_mb = (num(cdf.get("wal_lag_bytes")) or 0) / 1024**2
    level = (
        FAIL
        if not (cdf.get("sync_enabled") and streaming)
        else status_high(lag_mb, THRESHOLDS.wal_lag_mb_warn, THRESHOLDS.wal_lag_mb_fail)
    )
    return Outcome(
        level,
        f"sync={cdf.get('sync_enabled')}, wal2delta={wal[0]['status'] if wal else 'missing'}, lag {lag_mb:,.1f} MB",
        f"STREAMING; WAL lag WARN >= {THRESHOLDS.wal_lag_mb_warn} MB, FAIL >= {THRESHOLDS.wal_lag_mb_fail} MB",
        "Shadow-table CDF sync state from lakets._chronotable_registry and wal2delta.tables.",
    )


def _ops_job_rows() -> list[dict[str, Any]]:
    rows = []
    for kind in LAKETS_JOB_KINDS:
        name = f"LakeTS {kind} — {CTX.hot_project}/{CTX.hot_branch}"
        found = DATA.jobs(f"ops:{kind}", lambda n=name: BACKEND.jobs_named(n))
        last = (
            DATA.jobs(f"ops-run:{kind}", lambda f=found: BACKEND.runs(f[0]["job_id"], 3))
            if found
            else []
        )
        latest = last[0] if last else {}
        rows.append(
            {"job": f"LakeTS {kind}", "deployed": bool(found),
             "last run": fmt_ts(latest.get("start")) if latest else "-",
             "result": f"{latest.get('life_cycle')}/{latest.get('result') or '-'}" if latest else "-",
             "error": (latest.get("message") or "")[:100]}
        )  # fmt: skip
    return rows


def check_ops_jobs() -> Outcome:
    _hot()
    rows = _ops_job_rows()
    missing = [r["job"] for r in rows if not r["deployed"]]
    failed = [
        r["job"]
        for r in rows
        if str(r["result"]).endswith(("FAILED", "TIMEDOUT", "INTERNAL_ERROR"))
    ]
    level = FAIL if failed else WARN if missing else PASS
    return Outcome(
        level,
        f"{len(rows) - len(missing)}/{len(rows)} deployed, {len(failed)} failing",
        "all ops jobs deployed, last run not failed",
        f"Missing: {', '.join(missing) or 'none'}; failing: {', '.join(failed) or 'none'}.",
    )


def view_lakets_effect() -> None:
    _hot()
    chunks = list(_chunks().values())
    rows = [
        {"status": c["status"], "partitions": c["chunks"], "rows": c["rows"], "size": fmt_bytes(c["bytes"]),
         "oldest": fmt_ts(c["oldest"]), "newest": fmt_ts(c["newest"]), "last tiered": fmt_ts(c["last_tiered_at"])}
        for c in chunks
    ]  # fmt: skip
    emit_html(heading("Partition lifecycle (effect of tiering / retention)", 4) + html_table(rows))
    policies = DATA.lb("lk_policies")
    emit_html(heading("Registered policies", 4) + html_table(
        [{"type": p["policy_type"], "enabled": p["enabled"], "config": json.dumps(p["config"]),
          "last run": fmt_ts(p["last_run_at"])} for p in policies]))  # fmt: skip
    rollups = DATA.lb("lk_rollups")
    emit_html(heading("Rollups", 4) + html_table(
        [{"rollup": r["name"], "bucket": r["bucket"], "watermark": fmt_ts(r["watermark"]),
          "refreshed": fmt_ts(r["last_refreshed_at"]), "pending invalidations": r["pending_invalidations"]}
         for r in rollups]))  # fmt: skip
    emit_html(heading("LakeTS ops jobs (partition manager, rollup refresh, tiering, retention)", 4)
              + html_table(_ops_job_rows()))  # fmt: skip


emit_html(heading("4. LakeTS outcomes", 2) + sample_banner())
REPORT.run(SECTION_LAKETS, "lakets_installed", check_lakets_installed)
REPORT.run(SECTION_LAKETS, "lakets_partitions", check_lakets_partitions)
REPORT.run(SECTION_LAKETS, "lakets_tiering", check_lakets_tiering)
REPORT.run(SECTION_LAKETS, "lakets_rollups", check_lakets_rollups)
REPORT.run(SECTION_LAKETS, "lakets_lvc", check_lakets_lvc)
REPORT.run(SECTION_LAKETS, "lakets_gap_fill", check_gap_fill)
REPORT.run(SECTION_LAKETS, "lakets_rate_of_change", check_rate_of_change)
REPORT.run(SECTION_LAKETS, "lakets_value_range", check_value_range)
REPORT.run(SECTION_LAKETS, "lakets_cdf_state", check_lakets_cdf)
REPORT.run(SECTION_LAKETS, "lakets_ops_jobs", check_ops_jobs)
REPORT.view("LakeTS effect tables", view_lakets_effect)
finish_section(SECTION_LAKETS)

## 4. LakeTS outcomes

SAMPLE DATA — synthetic dataset, no workspace was contacted

status,partitions,rows,size,oldest,newest,last tiered
active,120,"1,540,000",39.1 GB,2026-11-07 09:17:09 UTC,2026-11-20 10:02:09 UTC,-
tiered,340,"6,100,000",0 B,2026-11-04 09:17:09 UTC,2026-11-07 09:17:09 UTC,2026-11-20 08:52:09 UTC
dropped,210,"3,300,000",0 B,2026-10-29 09:17:09 UTC,2026-11-04 09:17:09 UTC,2026-11-20 08:52:09 UTC


type,enabled,config,last run
tiered_retention,True,"{""tier_after"": ""7 days"", ""drop_after"": ""14 days""}",2026-11-20 08:52:09 UTC


rollup,bucket,watermark,refreshed,pending invalidations
sensor_1m,00:01:00,2026-11-20 09:15:09 UTC,2026-11-20 09:15:34 UTC,12


job,deployed,last run,result,error
LakeTS Partition Manager,True,2026-11-18 05:17:09 UTC,RUNNING/-,
LakeTS Rollup Refresh,True,2026-11-18 05:17:09 UTC,RUNNING/-,
LakeTS Tiering,True,2026-11-18 05:17:09 UTC,RUNNING/-,
LakeTS Retention,True,2026-11-18 05:17:09 UTC,RUNNING/-,


check,status,value,threshold,explanation
lakets_installed,PASS,0.1.0,lakets._version present,LakeTS 0.1.0 installed 2026-11-11 09:17:09 UTC.
lakets_partitions,PASS,"3 forward, 96 created in 24h",forward partitions >= 1,"Active 120, tiered 340, dropped 210 partitions."
lakets_tiering,PASS,"tiered_retention, last run 0.4h ago, 2 eligible",eligible chunks with last run > 6.0h = WARN,"Tiered 340, dropped 210 partitions so far; policy enabled."
lakets_rollups,PASS,"refresh age 1.6m, 12 pending",age WARN > 600s / FAIL > 3600s; pending WARN >= 100,1 rollup(s): sensor_1m
lakets_lvc,PASS,"lag 1s, 18 series","WARN > 120s, FAIL > 900s",Last-value cache event_time versus the source table (feeds the live dashboards).
lakets_gap_fill,WARN,41 of 540 (7.6%),"missing WARN >= 5.0%, FAIL >= 25.0%",Missing 1-minute buckets per sensor over the last 30 minutes (18 sensors).
lakets_rate_of_change,PASS,158 outliers (0.12%),"> 6.0 sigma; WARN >= 0.5%, FAIL >= 5.0%",Per-sensor rate of change over 2h; max |rate| 41.7/s.
lakets_value_range,PASS,"4,642 of 1,547,424 (0.30%)","WARN >= 1.0%, FAIL >= 10.0%","Value distribution (newest 6h): readings outside the [min_value, max_value] on the same row."
lakets_cdf_state,PASS,"sync=True, wal2delta=STREAMING, lag 3.1 MB","STREAMING; WAL lag WARN >= 64.0 MB, FAIL >= 1024.0 MB",Shadow-table CDF sync state from lakets._chronotable_registry and wal2delta.tables.
lakets_ops_jobs,PASS,"4/4 deployed, 0 failing","all ops jobs deployed, last run not failed",Missing: none; failing: none.


In [16]:
def _uc_ready() -> None:
    if not CTX.deployment:
        raise CheckSkipped("NOT FOUND: pipeline was not resolved")
    if not CTX.can_query_uc:
        raise CheckSkipped(
            "NOT FOUND: catalog/schema could not be derived (set uc_catalog / uc_schema)"
        )


def check_uc_connect() -> Outcome:
    _uc_ready()
    tables = DATA.uc(
        "uc_tables", catalog=uc_ident(CTX.catalog), schema=uc_lit(CTX.schema), industry=uc_lit(PARAMS.industry),
        history=uc_lit(CTX.history_table),
    )  # fmt: skip
    return Outcome(
        PASS, f"{len(tables)} tables listed", "information_schema readable",
        "Unity Catalog SQL is reachable (warehouse or notebook compute).",
    )  # fmt: skip


def _tables() -> dict[str, dict[str, Any]]:
    rows = DATA.uc(
        "uc_tables", catalog=uc_ident(CTX.catalog), schema=uc_lit(CTX.schema), industry=uc_lit(PARAMS.industry),
        history=uc_lit(CTX.history_table),
    )  # fmt: skip
    return {r["table_name"]: r for r in rows}


def check_uc_tables() -> Outcome:
    _uc_ready()
    tables = _tables()
    expected = (
        {CTX.cold_table, CTX.metrics_table}
        if PARAMS.mode in SPARK_METRIC_MODES
        else {CTX.cold_table}
    )
    missing = sorted(expected - set(tables))
    return Outcome(
        FAIL if missing else PASS,
        f"{len(expected) - len(missing)}/{len(expected)} present",
        "all mode tables exist",
        f"Missing tables: {', '.join(missing) or 'none'}.",
    )


def _window() -> dict[str, Any]:
    _uc_ready()
    return DATA.uc_one("uc_window", table=CTX.uc_table(), hours=int(PARAMS.lookback_hours))


def check_uc_rows() -> Outcome:
    window = _window()
    n = int(window["n"] or 0)
    return Outcome(
        PASS if n >= THRESHOLDS.min_rows_in_window else FAIL,
        fmt_num(n),
        f">= {THRESHOLDS.min_rows_in_window} rows in {PARAMS.lookback_hours}h",
        f"{fmt_num(n)} rows, {fmt_num(window['sensors'])} sensors, "
        f"{fmt_ts(window['min_t'])} to {fmt_ts(window['max_t'])}.",
    )


def check_uc_freshness() -> Outcome:
    lag = num(_window()["lag_s"])
    warn, fail = THRESHOLDS.freshness(PARAMS.mode)
    if PARAMS.mode == "realtime":  # CDF adds flush latency on top of the hot tier
        warn, fail = THRESHOLDS.cdf_lag_s_warn, THRESHOLDS.cdf_lag_s_fail
    return Outcome(
        status_high(lag, warn, fail),
        fmt_seconds(lag),
        f"WARN > {warn}s, FAIL > {fail}s",
        f"Newest event_time in the cold-tier table is {fmt_seconds(lag)} old.",
    )


def _history() -> list[dict[str, Any]]:
    _uc_ready()
    return DATA.uc("uc_history", table=CTX.uc_table())


def check_uc_last_commit() -> Outcome:
    history = _history()
    if not history:
        return Outcome(FAIL, "no history", "recent commit", "DESCRIBE HISTORY returned no commits.")
    latest = to_dt(history[0]["timestamp"])
    age = (datetime.now(UTC) - latest).total_seconds() if latest else None
    return Outcome(
        status_high(age, THRESHOLDS.delta_last_commit_s_warn, THRESHOLDS.delta_last_commit_s_fail),
        f"{fmt_seconds(age)} ago ({history[0]['operation']})",
        f"WARN > {THRESHOLDS.delta_last_commit_s_warn}s, FAIL > {THRESHOLDS.delta_last_commit_s_fail}s",
        f"Latest Delta version {history[0]['version']}.",
    )


def check_small_files() -> Outcome:
    _uc_ready()
    detail = DATA.uc_one("uc_detail", table=CTX.uc_table())
    files, size = int(detail["numFiles"] or 0), num(detail["sizeInBytes"]) or 0
    avg_mb = size / files / 1024**2 if files else None
    small = files >= THRESHOLDS.small_file_min_files and avg_mb is not None
    level = WARN if small and avg_mb < THRESHOLDS.small_file_avg_mb_warn else PASS
    return Outcome(
        level,
        f"{fmt_num(files)} files, avg {avg_mb:,.1f} MB" if avg_mb is not None else "0 files",
        f"WARN: >= {THRESHOLDS.small_file_min_files} files and avg < {THRESHOLDS.small_file_avg_mb_warn} MB",
        f"Total {fmt_bytes(size)}.",
    )


def _cdf_binding() -> tuple[bool, str, bool]:
    wal = DATA.lb("lk_wal2delta", shadow=CDF_SHADOW_TABLE) if CTX.hot_ready else []
    streaming = bool(wal) and str(wal[0]["status"]).upper() == "STREAMING"
    ledger = any(r.get("resource_type") == "cdf_sync" for r in CTX.resources)
    return streaming, (wal[0]["status"] if wal else "unknown"), ledger


def check_cdf_reconciliation_state() -> Outcome:
    if not CTX.has_hot_tier:
        raise NotApplicable("CDF reconciliation is a realtime-only concept")
    if not CTX.deployment:
        raise CheckSkipped("NOT FOUND: pipeline was not resolved")
    streaming, status, ledger = _cdf_binding()
    enabled = bool(CTX.deployment.get("cdf_enabled"))
    ok = streaming and ledger and enabled
    return Outcome(
        PASS if ok else FAIL,
        f"enabled={enabled}, ledger={ledger}, wal2delta={status}",
        "cdf_enabled, cdf_sync ledger row, wal2delta STREAMING",
        "Control-plane reconciliation state (services/cdf_reconciliation_service) agrees with the database.",
    )


def check_hot_cold_lag() -> Outcome:
    if not CTX.has_hot_tier:
        raise NotApplicable("only realtime has a Lakebase tier to compare with")
    _hot()
    hot = DATA.lb_one("lb_latest")
    cold = _window()
    lag = (to_dt(hot["max_t"]) - to_dt(cold["max_t"])).total_seconds()
    return Outcome(
        status_high(lag, THRESHOLDS.cdf_lag_s_warn, THRESHOLDS.cdf_lag_s_fail),
        fmt_seconds(lag),
        f"WARN > {THRESHOLDS.cdf_lag_s_warn}s, FAIL > {THRESHOLDS.cdf_lag_s_fail}s",
        "Newest event_time in Lakebase minus newest event_time in Delta (CDF replication lag).",
    )


def view_cold_tables() -> None:
    _uc_ready()
    tables = _tables()
    expected = [CTX.history_table, f"{PARAMS.industry}_realtime", f"{PARAMS.industry}_microbatch",
                f"{PARAMS.industry}_zerobus", CTX.metrics_table]  # fmt: skip
    rows = [
        {"table": name, "exists": name in tables, "type": (tables.get(name) or {}).get("table_type", "-"),
         "last altered": fmt_ts((tables.get(name) or {}).get("last_altered"))}
        for name in expected
    ]  # fmt: skip
    emit_html(heading("Industry tables", 4) + html_table(rows))
    emit_html(heading(f"Cold-tier table in focus: {CTX.cold_table}", 4) + kv_table([
        ("rows in window", fmt_num(_window()["n"])), ("freshness lag", fmt_seconds(_window()["lag_s"]))]))  # fmt: skip


def view_cold_history() -> None:
    history = _history()
    rows = [
        {"version": h["version"], "timestamp": fmt_ts(h["timestamp"]), "operation": h["operation"],
         "files": h.get("numFiles"), "output rows": h.get("numOutputRows")}
        for h in history[:10]
    ]  # fmt: skip
    emit_html(heading("Delta history (latest commits)", 4) + html_table(rows))
    series = DATA.uc("uc_rate_hour", table=CTX.uc_table(), hours=int(PARAMS.lookback_hours))
    emit_html(
        svg_bar_chart(
            [(fmt_ts(r["bucket"])[5:16], float(r["n"])) for r in series],
            title=f"Cold-tier rows per hour - last {PARAMS.lookback_hours}h",
            y_label="rows per hour",
        )
    )


emit_html(heading("5. CDF / Unity Catalog cold tier", 2) + sample_banner())
REPORT.run(SECTION_COLD, "uc_connect", check_uc_connect)
REPORT.run(SECTION_COLD, "uc_tables", check_uc_tables)
REPORT.run(SECTION_COLD, "uc_rows_window", check_uc_rows)
REPORT.run(SECTION_COLD, "uc_freshness", check_uc_freshness)
REPORT.run(SECTION_COLD, "uc_last_commit", check_uc_last_commit)
REPORT.run(SECTION_COLD, "uc_small_files", check_small_files)
REPORT.run(SECTION_COLD, "uc_cdf_state", check_cdf_reconciliation_state)
REPORT.run(SECTION_COLD, "uc_cdf_lag", check_hot_cold_lag)
REPORT.view("cold-tier tables", view_cold_tables)
REPORT.view("delta history", view_cold_history)
finish_section(SECTION_COLD)

## 5. CDF / Unity Catalog cold tier

SAMPLE DATA — synthetic dataset, no workspace was contacted

table,exists,type,last altered
lb__shadow_manufacturing_sensor_readings_history,True,MANAGED,2026-11-20 09:15:09 UTC
manufacturing_realtime,False,-,-
manufacturing_microbatch,True,MANAGED,2026-11-20 09:13:09 UTC
manufacturing_zerobus,True,MANAGED,2026-11-20 09:12:09 UTC
manufacturing_metrics,True,MANAGED,2026-11-20 09:14:09 UTC


field,value
rows in window,"1,454,578"
freshness lag,7.2m


version,timestamp,operation,files,output rows
412,2026-11-20 09:15:44 UTC,STREAMING UPDATE,2,"1,500"
411,2026-11-20 09:10:44 UTC,STREAMING UPDATE,3,"1,520"
410,2026-11-20 09:05:44 UTC,OPTIMIZE,4,"1,540"
409,2026-11-20 09:00:44 UTC,STREAMING UPDATE,2,"1,560"
408,2026-11-20 08:55:44 UTC,STREAMING UPDATE,3,"1,580"
407,2026-11-20 08:50:44 UTC,STREAMING UPDATE,4,"1,600"
406,2026-11-20 08:45:44 UTC,OPTIMIZE,2,"1,620"
405,2026-11-20 08:40:44 UTC,STREAMING UPDATE,3,"1,640"
404,2026-11-20 08:35:44 UTC,STREAMING UPDATE,4,"1,660"
403,2026-11-20 08:30:44 UTC,STREAMING UPDATE,2,"1,680"


check,status,value,threshold,explanation
uc_connect,PASS,4 tables listed,information_schema readable,Unity Catalog SQL is reachable (warehouse or notebook compute).
uc_tables,PASS,2/2 present,all mode tables exist,Missing tables: none.
uc_rows_window,PASS,"1,454,578",>= 1 rows in 24h,"1,454,578 rows, 18 sensors, 2026-11-19 09:17:09 UTC to 2026-11-20 09:09:59 UTC."
uc_freshness,WARN,7.2m,"WARN > 300s, FAIL > 1800s",Newest event_time in the cold-tier table is 7.2m old.
uc_last_commit,PASS,86s ago (STREAMING UPDATE),"WARN > 900s, FAIL > 3600s",Latest Delta version 412.
uc_small_files,WARN,"1,840 files, avg 4.8 MB",WARN: >= 50 files and avg < 16.0 MB,Total 8.6 GB.
uc_cdf_state,PASS,"enabled=True, ledger=True, wal2delta=STREAMING","cdf_enabled, cdf_sync ledger row, wal2delta STREAMING",Control-plane reconciliation state (services/cdf_reconciliation_service) agrees with the database.
uc_cdf_lag,WARN,7.1m,"WARN > 300s, FAIL > 1800s",Newest event_time in Lakebase minus newest event_time in Delta (CDF replication lag).


In [17]:
def pctl(values: list[float], q: float) -> float | None:
    """Nearest-rank percentile (q in 0..100) of a non-empty list; None when empty."""
    ordered = sorted(v for v in values if v is not None)
    if not ordered:
        return None
    return ordered[min(len(ordered) - 1, max(0, math.ceil(q / 100 * len(ordered)) - 1))]


def _metrics_rows() -> list[dict[str, Any]]:
    if PARAMS.mode not in SPARK_METRIC_MODES:
        raise NotApplicable("zerobus is not a Spark stream - it has no streaming-metrics table")
    _uc_ready()
    return DATA.uc(
        "uc_metrics", table=CTX.uc_table(CTX.metrics_table), mode=PARAMS.mode,
        hours=int(PARAMS.lookback_hours),
    )  # fmt: skip


def _column(rows: list[dict[str, Any]], name: str) -> list[float]:
    return [v for v in (num(r.get(name)) for r in rows) if v is not None]


def check_metrics_table() -> Outcome:
    _uc_ready()
    if PARAMS.mode not in SPARK_METRIC_MODES:
        raise NotApplicable("zerobus has no <industry>_metrics rows (see the zerobus ingest check)")
    rows = _metrics_rows()
    return Outcome(
        PASS if rows else WARN,
        f"{len(rows)} batches",
        f">= 1 row for mode='{PARAMS.mode}' in {PARAMS.lookback_hours}h",
        f"{CTX.metrics_table} holds {len(rows)} metric rows for this mode in the window.",
    )


def check_metrics_freshness() -> Outcome:
    rows = _metrics_rows()
    if not rows:
        raise CheckSkipped("NOT FOUND: no metric rows in the window")
    latest = max(to_dt(r["timestamp"]) for r in rows)
    age = (datetime.now(UTC) - latest).total_seconds()
    return Outcome(
        status_high(age, THRESHOLDS.metrics_fresh_s_warn, THRESHOLDS.metrics_fresh_s_fail),
        fmt_seconds(age),
        f"WARN > {THRESHOLDS.metrics_fresh_s_warn}s, FAIL > {THRESHOLDS.metrics_fresh_s_fail}s",
        f"Latest batch {fmt_ts(latest)}.",
    )


def check_e2e_latency() -> Outcome:
    values = _column(_metrics_rows(), "e2e_p99")
    p95 = pctl(values, 95)
    return Outcome(
        status_high(p95, THRESHOLDS.e2e_p99_ms_warn, THRESHOLDS.e2e_p99_ms_fail),
        f"p95 of batch p99 = {fmt_ms(p95)}",
        f"WARN > {fmt_ms(THRESHOLDS.e2e_p99_ms_warn)}, FAIL > {fmt_ms(THRESHOLDS.e2e_p99_ms_fail)}",
        f"Spark source-available to trigger-complete latency (median p50 {fmt_ms(pctl(_column(_metrics_rows(), 'e2e_p50'), 50))}).",
    )


def check_batch_duration() -> Outcome:
    p95 = pctl(_column(_metrics_rows(), "trigger_ms"), 95)
    return Outcome(
        status_high(p95, THRESHOLDS.batch_ms_warn, THRESHOLDS.batch_ms_fail),
        f"p95 = {fmt_ms(p95)}",
        f"WARN > {fmt_ms(THRESHOLDS.batch_ms_warn)}, FAIL > {fmt_ms(THRESHOLDS.batch_ms_fail)}",
        "Trigger execution time per batch across the window.",
    )


def check_throughput() -> Outcome:
    rows = _metrics_rows()
    inp, proc = _column(rows, "input_rps"), _column(rows, "processed_rps")
    if not inp or not proc or sum(inp) == 0:
        raise CheckSkipped("NOT FOUND: no throughput samples")
    ratio = (sum(proc) / len(proc)) / (sum(inp) / len(inp))
    return Outcome(
        PASS if ratio >= THRESHOLDS.lag_behind_ratio_warn else WARN,
        f"processed/input = {ratio:.2f} ({sum(proc) / len(proc):,.1f} vs {sum(inp) / len(inp):,.1f} rows/s)",
        f"WARN < {THRESHOLDS.lag_behind_ratio_warn}",
        "Average processed rows/s relative to input rows/s (below 1 means the stream is falling behind).",
    )


def check_zerobus_ingest() -> Outcome:
    if PARAMS.mode != "zerobus":
        raise NotApplicable("zerobus ingest system table only applies to the zerobus mode")
    _uc_ready()
    name = ".".join(uc_lit(p) for p in (CTX.catalog, CTX.schema, CTX.cold_table))
    summary = DATA.uc_one("uc_zerobus_summary", table_fqn=name, hours=int(PARAMS.lookback_hours))
    lag, errors = num(summary["seconds_since_commit"]), int(summary["error_count"] or 0)
    warn, fail = THRESHOLDS.freshness("zerobus")
    level = max(
        (
            status_high(lag, warn, fail),
            status_high(errors, THRESHOLDS.zerobus_errors_warn, THRESHOLDS.zerobus_errors_fail),
        ),
        key=lambda s: {PASS: 0, SKIPPED: 0, WARN: 1, FAIL: 2}[s],
    )
    per_min = int(summary["committed_records"] or 0) / (PARAMS.lookback_hours * 60)
    return Outcome(
        level,
        f"last commit {fmt_seconds(lag)} ago, {errors} errors, {per_min:,.0f} rec/min",
        f"commit age WARN > {warn}s / FAIL > {fail}s; errors WARN >= {THRESHOLDS.zerobus_errors_warn}",
        "system.lakeflow.zerobus_ingest for the zerobus target table.",
    )


def view_latency_charts() -> None:
    rows = sorted(_metrics_rows(), key=lambda r: to_dt(r["timestamp"]))
    if not rows:
        raise CheckSkipped("NOT FOUND: no metric rows to chart")

    def line(col: str) -> list[tuple[float, float]]:
        return [
            (to_dt(r["timestamp"]).timestamp(), float(r[col]))
            for r in rows
            if r.get(col) is not None
        ]

    ends = (fmt_ts(rows[0]["timestamp"]), fmt_ts(rows[-1]["timestamp"]))
    emit_html(svg_line_chart({"e2e p50": line("e2e_p50"), "e2e p99": line("e2e_p99"),
                              "processing p99": line("processing_p99")},
                             title="Latency percentiles (ms)", y_label="ms", x_ends=ends,
                             hline=(THRESHOLDS.e2e_p99_ms_warn, "e2e p99 warn")))  # fmt: skip
    emit_html(svg_line_chart({"input rows/s": line("input_rps"), "processed rows/s": line("processed_rps")},
                             title="Throughput", y_label="rows/s", x_ends=ends))  # fmt: skip
    emit_html(svg_line_chart({"trigger ms": line("trigger_ms")}, title="Batch duration (trigger execution)",
                             y_label="ms", x_ends=ends, hline=(THRESHOLDS.batch_ms_warn, "warn")))  # fmt: skip


def view_zerobus_series() -> None:
    if PARAMS.mode != "zerobus":
        raise NotApplicable("zerobus only")
    _uc_ready()
    name = ".".join(uc_lit(p) for p in (CTX.catalog, CTX.schema, CTX.cold_table))
    series = DATA.uc("uc_zerobus_series", table_fqn=name, hours=int(PARAMS.lookback_hours))
    emit_html(svg_bar_chart([(fmt_ts(r["bucket"])[5:16], float(r["records"])) for r in series],
                            title="Zerobus committed records per hour", y_label="records"))  # fmt: skip


emit_html(heading("6. Streaming metrics", 2) + sample_banner())
REPORT.run(SECTION_METRICS, "metrics_table", check_metrics_table)
REPORT.run(SECTION_METRICS, "metrics_freshness", check_metrics_freshness)
REPORT.run(SECTION_METRICS, "metrics_e2e_latency", check_e2e_latency)
REPORT.run(SECTION_METRICS, "metrics_batch_duration", check_batch_duration)
REPORT.run(SECTION_METRICS, "metrics_throughput", check_throughput)
REPORT.run(SECTION_METRICS, "zerobus_ingest", check_zerobus_ingest)
REPORT.view("latency/throughput charts", view_latency_charts)
REPORT.view("zerobus series", view_zerobus_series)
finish_section(SECTION_METRICS)

## 6. Streaming metrics

SAMPLE DATA — synthetic dataset, no workspace was contacted

zerobus series: SKIPPED - zerobus only

check,status,value,threshold,explanation
metrics_table,PASS,288 batches,>= 1 row for mode='realtime' in 24h,manufacturing_metrics holds 288 metric rows for this mode in the window.
metrics_freshness,PASS,21s,"WARN > 600s, FAIL > 1800s",Latest batch 2026-11-20 09:16:49 UTC.
metrics_e2e_latency,PASS,"p95 of batch p99 = 1,718 ms","WARN > 5,000 ms, FAIL > 30.0 s",Spark source-available to trigger-complete latency (median p50 265 ms).
metrics_batch_duration,PASS,"p95 = 2,287 ms","WARN > 10.0 s, FAIL > 60.0 s",Trigger execution time per batch across the window.
metrics_throughput,PASS,processed/input = 1.02 (18.4 vs 18.0 rows/s),WARN < 0.9,Average processed rows/s relative to input rows/s (below 1 means the stream is falling behind).
zerobus_ingest,SKIPPED,-,-,NOT APPLICABLE: zerobus ingest system table only applies to the zerobus mode


In [18]:
from datetime import timedelta as _td


def _norm_type(value: Any) -> str:
    return re.sub(r"\s+", "", str(value)).lower()


def grade_schema(
    expected: dict[str, str], actual: dict[str, str], ignore: tuple[str, ...]
) -> Outcome:
    """FAIL on missing columns, WARN on type drift or unexpected extras."""
    missing = sorted(set(expected) - set(actual))
    extra = sorted(set(actual) - set(expected) - set(ignore))
    drift = sorted(
        c for c in expected if c in actual and _norm_type(expected[c]) != _norm_type(actual[c])
    )
    level = FAIL if missing else WARN if (extra or drift) else PASS
    detail = f"missing: {missing or 'none'}; unexpected: {extra or 'none'}; type drift: {drift or 'none'}"
    return Outcome(level, f"{len(missing)} missing, {len(extra)} extra, {len(drift)} drifted",
                   "columns and types equal the sensor SSOT", detail)  # fmt: skip


def check_schema_hot() -> Outcome:
    _hot()
    expected = {c: pg for c, pg, _d, _n in EXPECTED_SENSOR_COLUMNS}
    actual = {r["column_name"]: r["data_type"] for r in DATA.lb("lb_columns")}
    return grade_schema(expected, actual, PG_ONLY_COLUMNS)


def check_schema_cold() -> Outcome:
    _uc_ready()
    expected = {c: d for c, _p, d, _n in EXPECTED_SENSOR_COLUMNS}
    rows = DATA.uc("uc_columns", catalog=uc_ident(CTX.catalog), schema=uc_lit(CTX.schema),
                   table_name=uc_lit(CTX.cold_table))  # fmt: skip
    return grade_schema(expected, {r["column_name"]: r["data_type"] for r in rows}, ())


def check_reconciliation() -> Outcome:
    _hot()
    _uc_ready()
    hot_now = to_dt(DATA.lb_one("lb_latest")["db_now"])
    hours = scan_hours()
    t1 = (hot_now - _td(minutes=THRESHOLDS.recon_settle_min)).replace(microsecond=0)
    t0 = (t1 - _td(hours=hours)).replace(microsecond=0)
    hot = int(DATA.lb_one("lb_recon", t0=t0.isoformat(), t1=t1.isoformat())["n"] or 0)
    stamp0, stamp1 = (t.strftime("%Y-%m-%d %H:%M:%S") for t in (t0, t1))
    cold = int(DATA.uc_one("uc_recon", table=CTX.uc_table(), t0=stamp0, t1=stamp1)["n"] or 0)
    diff_pct = 100 * abs(hot - cold) / hot if hot else None
    return Outcome(
        status_high(diff_pct, THRESHOLDS.recon_diff_pct_warn, THRESHOLDS.recon_diff_pct_fail),
        f"Lakebase {fmt_num(hot)} vs Delta {fmt_num(cold)} ({fmt_pct(diff_pct)})",
        f"WARN >= {THRESHOLDS.recon_diff_pct_warn}%, FAIL >= {THRESHOLDS.recon_diff_pct_fail}%",
        f"Same closed {hours}h window [{stamp0}, {stamp1}) UTC; newest {THRESHOLDS.recon_settle_min} min excluded "
        "so CDF flush latency does not count as loss. Rows tiered/dropped from Lakebase in the "
        "window would also show up here.",
    )


def check_cold_duplicates() -> Outcome:
    _uc_ready()
    dupes = DATA.uc_one("uc_dupes", table=CTX.uc_table(), qhours=int(QUALITY_HOURS))
    pct = 100 * int(dupes["dup_rows"] or 0) / max(int(dupes["n"] or 0), 1)
    return Outcome(
        status_high(pct, THRESHOLDS.duplicate_pct_warn, THRESHOLDS.duplicate_pct_fail),
        f"{fmt_num(dupes['dup_rows'])} rows ({pct:.4f}%)",
        f"WARN >= {THRESHOLDS.duplicate_pct_warn}%, FAIL >= {THRESHOLDS.duplicate_pct_fail}%",
        f"Repeated (sensor_name, event_time, value) triples in the newest {QUALITY_HOURS}h.",
    )


def check_cold_nulls() -> Outcome:
    _uc_ready()
    row = DATA.uc_one("uc_nulls", table=CTX.uc_table(), qhours=int(QUALITY_HOURS))
    return grade_nulls(_null_pcts(row), QUALITY_HOURS)


def check_e2e_estimate() -> Outcome:
    """Serving-tier delay: Lakebase lag for realtime, Delta lag for the Delta-only modes."""
    warn, fail = THRESHOLDS.freshness(PARAMS.mode)
    if CTX.has_hot_tier:
        _hot()
        serving = num(DATA.lb_one("lb_latest")["lag_s"])
        tier = "Lakebase"
    else:
        serving, tier = num(_window()["lag_s"]), "Delta"
    return Outcome(
        status_high(serving, warn, fail),
        f"{fmt_seconds(serving)} to {tier}",
        f"WARN > {warn}s, FAIL > {fail}s",
        f"Estimated event-time to {tier} delay (newest row age, includes producer + Kafka + sink).",
    )


def check_sse_evidence() -> Outcome:
    """Indirect evidence that the app's live feed (SSE) has data: LVC cache + alert events."""
    if not CTX.has_hot_tier:
        raise NotApplicable("the SSE dual-write feed is realtime-only")
    _hot()
    lvc = DATA.lb_one("lk_lvc")
    alerts = DATA.lb_one("cp_alert_events", industry=PARAMS.industry, hours=PARAMS.lookback_hours)
    lag = num(lvc["lag_s"])
    level = status_high(lag, THRESHOLDS.lvc_lag_s_warn, THRESHOLDS.lvc_lag_s_fail)
    return Outcome(
        level,
        f"LVC lag {fmt_seconds(lag)}, {alerts['events']} alert events",
        f"LVC lag WARN > {THRESHOLDS.lvc_lag_s_warn}s",
        "The SSE stream itself is not persisted; this uses the last-value cache the dashboards read "
        f"and alert events derived from the stream (last at {fmt_ts(alerts['last_event_at'])}).",
    )


def view_latency_breakdown() -> None:
    rows: list[dict[str, Any]] = []
    if CTX.has_hot_tier:
        _hot()
        rows.append({"hop": "event -> Lakebase (newest row age)",
                     "value": fmt_seconds(DATA.lb_one("lb_latest")["lag_s"])})  # fmt: skip
    if CTX.can_query_uc:
        rows.append(
            {"hop": "event -> Delta (newest row age)", "value": fmt_seconds(_window()["lag_s"])}
        )
    if PARAMS.mode in SPARK_METRIC_MODES and CTX.can_query_uc:
        values = _column(_metrics_rows(), "e2e_p50")
        rows.append(
            {
                "hop": "Spark source -> trigger complete (median p50)",
                "value": fmt_ms(pctl(values, 50)),
            }
        )
    emit_html(heading("End-to-end latency estimate by hop", 4) + html_table(rows))


emit_html(heading("7. Cross-checks & data quality", 2) + sample_banner())
REPORT.run(SECTION_CROSS, "xc_reconciliation", check_reconciliation)
REPORT.run(SECTION_CROSS, "xc_schema_hot", check_schema_hot)
REPORT.run(SECTION_CROSS, "xc_schema_cold", check_schema_cold)
REPORT.run(SECTION_CROSS, "xc_duplicates_cold", check_cold_duplicates)
REPORT.run(SECTION_CROSS, "xc_nulls_cold", check_cold_nulls)
REPORT.run(SECTION_CROSS, "xc_e2e_latency", check_e2e_estimate)
REPORT.run(SECTION_CROSS, "xc_sse_evidence", check_sse_evidence)
REPORT.view("latency breakdown", view_latency_breakdown)
finish_section(SECTION_CROSS)

## 7. Cross-checks & data quality

SAMPLE DATA — synthetic dataset, no workspace was contacted

hop,value
event -> Lakebase (newest row age),3s
event -> Delta (newest row age),7.2m
Spark source -> trigger complete (median p50),265 ms


check,status,value,threshold,explanation
xc_reconciliation,FAIL,"Lakebase 1,531,305 vs Delta 1,436,364 (6.20%)","WARN >= 1.0%, FAIL >= 5.0%","Same closed 24h window [2026-11-19 09:02:09, 2026-11-20 09:02:09) UTC; newest 15 min excluded so CDF flush latency does not count as loss. Rows tiered/dropped from Lakebase in the window would also show up here."
xc_schema_hot,PASS,"0 missing, 0 extra, 0 drifted",columns and types equal the sensor SSOT,missing: none; unexpected: none; type drift: none
xc_schema_cold,PASS,"0 missing, 0 extra, 0 drifted",columns and types equal the sensor SSOT,missing: none; unexpected: none; type drift: none
xc_duplicates_cold,PASS,0 rows (0.0000%),"WARN >= 0.01%, FAIL >= 1.0%","Repeated (sensor_name, event_time, value) triples in the newest 6h."
xc_nulls_cold,PASS,"required 0.000%, optional max 0.000%","required = 0%, optional WARN >= 0.1%, FAIL >= 5.0%",Columns with NULLs in the newest 6h: none.
xc_e2e_latency,PASS,3s to Lakebase,"WARN > 120s, FAIL > 600s","Estimated event-time to Lakebase delay (newest row age, includes producer + Kafka + sink)."
xc_sse_evidence,PASS,"LVC lag 1s, 31 alert events",LVC lag WARN > 120s,The SSE stream itself is not persisted; this uses the last-value cache the dashboards read and alert events derived from the stream (last at 2026-11-20 09:00:09 UTC).


In [19]:
_STATUS_ORDER = {FAIL: 0, WARN: 1, SKIPPED: 2, PASS: 3}


def render_scorecard() -> str:
    overall, counts = REPORT.overall(), REPORT.counts()
    overall_label = "NOT EVALUATED" if overall == SKIPPED else overall
    not_applicable = sum("NOT APPLICABLE" in r.explanation for r in REPORT.results)
    summary = " | ".join(f"{counts[s]} {s}" for s in (PASS, WARN, FAIL, SKIPPED))
    summary += f" ({not_applicable} not applicable to this mode)"
    banner = (
        f'<div style="font-size:18px;margin:6px 0">Overall: {badge(overall_label)} '
        f"&nbsp;<span style='opacity:.8'>{escape(summary)}</span></div>"
    )
    ordered = sorted(REPORT.results, key=lambda r: (_STATUS_ORDER[r.status], r.section, r.name))
    rows = [
        {"section": r.section, "check": r.name, "status": raw(badge(r.status)), "value": r.value,
         "threshold": r.threshold, "explanation": r.explanation}
        for r in ordered
    ]  # fmt: skip
    steps: dict[str, list[str]] = {}
    for r in ordered:
        if r.status in (FAIL, WARN) and r.next_step:
            steps.setdefault(r.next_step, []).append(r.name)
    step_rows = [{"#": i, "recommended next step": text, "because": ", ".join(names)}
                 for i, (text, names) in enumerate(steps.items(), 1)]  # fmt: skip
    advice = (
        html_table(step_rows, ["#", "recommended next step", "because"])
        if steps
        else note("No WARN/FAIL results - nothing to do.", "PASS")
    )
    return (
        heading("Scorecard", 2)
        + sample_banner()
        + banner
        + html_table(
            rows, ["section", "check", "status", "value", "threshold", "explanation"], max_rows=200
        )
        + heading("Recommended next steps", 3)
        + advice
        + note(
            f"Thresholds are defined in the configuration cell. Run took "
            f"{(datetime.now(UTC) - RUN_STARTED).total_seconds():.0f}s."
        )
    )


SCORECARD.update(render_scorecard())
HEADER.update(render_header(final=True))
BACKEND.close()
emit_html(
    heading("8. Final scorecard", 2) + sample_banner()
    + note(f"Overall {REPORT.overall()}. The scorecard at the top of the notebook was updated in "
           "place (Databricks cannot update earlier cells, so it is rendered above instead).")
)  # fmt: skip

## 8. Final scorecard

SAMPLE DATA — synthetic dataset, no workspace was contacted

Overall FAIL. The scorecard at the top of the notebook was updated in place (Databricks cannot update earlier cells, so it is rendered above instead).